<a href="https://colab.research.google.com/github/CHIN-HUA/AI/blob/main/%E3%80%8CA2_415511001_ipynb%E3%80%8D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 2 — The Morning Rush
## Official-data solution / 使用正式資料的獨立實作

**本 Notebook 包含 Part I 完整推導、數值驗算，以及 Part II–III 的可重現實驗流程。**
講義第 1 頁允許 Part I **手寫或打字排版**；本解答採打字排版，仍完整列出推導與計算。
正式檔名為 `A2_415511001.ipynb` 與 `A2_415511001.pdf`；Colab 連結仍待填入。

目前已收到作業講義、2026-10-19 的助教補充公告、Kaggle／繳交規則公告，
以及競賽頁面的資料欄位與評分說明。正式 ZIP 中的 `train.csv`、`test.csv`、
`sample_submission.csv` 與 `baseline.txt` 均已取得。
執行下方儲存格會載入與資料、程式版本相符的已存實驗結果，或重新執行正式實驗。
使用者提供的 Kaggle 截圖已確認隊名 **415511001_嚴佳華**、test RMSE **12.37017**、
截圖當時排名 **第 6 名**，並顯示 1 次提交。頁面註明排行榜使用全部測試資料；
這是截圖紀錄，並非即時排名。**12.37017 < 24.2461**，已符合排行榜的嚴格 baseline 門檻。

**配套資料：**三個 CSV 是訓練、驗證及預測所必需；`baseline.txt` 是官方
baseline 參考。已核對其排行榜門檻為 **test RMSE = 24.2461**；
本地 May validation 分數不能代替此模型的 Kaggle test 分數。

**助教補充公告：**starter notebook 為選用，已使用自寫程式的同學可繼續沿用。
本 Notebook 可作為獨立實作，不需要改用或合併官方 starter。內建學習曲線
仍會比較不同訓練資料量的 training / validation RMSE，並完成模型診斷與改善。

**提交前必要事項：**學號、姓名、隊名及實際 leaderboard 分數已依截圖核對；
目前仍需補上可檢視的 **Colab 連結**。Kaggle Team Name 須使用 `StudentID_Name`。
若已提交預測，只需更新隊名，不必重新提交預測。

### 最新公告的繳交規則

| 項目 | 規則 |
|---|---|
| Kaggle 用途 | Part III 的預測提交與排行榜評分 |
| 班級邀請連結 | https://www.kaggle.com/t/93a788c887e34269939b30da23305dfe |
| 截止時間 | **2026 年 11 月 6 日 23:59**，依課程公告時間 |
| 提交次數 | **每天最多 5 次** |
| 遲交 1 分鐘至 1 天 | 扣 **25%** |
| 遲交 1 天至 2 天 | 扣 **50%** |
| 遲交超過 2 天 | 扣 **100%** |

遲交規則依本次公告從 **1 分鐘**起算。先用 5 月驗證集選模，再提交已決定的
方案；每日提交額度不作為反覆使用測試分數調參的理由。

### 使用方式
1. 在 Colab 開啟此檔；學號、姓名與 Kaggle 分數已填入，請補上下一格的 Colab 連結。
2. 解壓配套 ZIP，讓 `data/`、`a2_results/` 和 Notebook 位於同一層；Colab 可將這些目錄放在 `/content`。也可另行上傳四個原始資料檔。
3. 執行全部儲存格；預設會核對資料與程式的 SHA-256，再載入已保存的正式輸出。要完整重訓，將 `REUSE_SAVED_RESULTS=False`。訓練期仍是 1–4 月、驗證期 5 月，最後以 1–5 月重訓預測 6 月。
4. 如希望另外使用老師的 helper，可選擇傳入；預設使用本程式的完整小時時間切分曲線。
5. 讀取下方顯示的 `baseline.txt` 原文；`TA_BASELINE_RMSE=24.2461` 已依此檔核對填入，程式不會盲目擷取或覆寫分數。
6. 已有 1 次提交的截圖紀錄，隊名為 `415511001_嚴佳華`，分數為 `12.37017`。若之後改變預測檔並重新提交，需在期限與每日額度內操作，並以對應的新成績更新設定。
7. 補入可檢視的 Colab 連結後重新產生報告。只更正隊名時不必重新上傳預測。

若只搬移 Notebook 而未放入資料，正式實驗會清楚跳過，Part I 仍可直接執行。


In [ ]:
from pathlib import Path
import sys, importlib.util, random, hashlib, json
import numpy as np
import pandas as pd
import sklearn

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except (ModuleNotFoundError, ValueError):
    IN_COLAB = False
BASE_DIR = Path('/content') if IN_COLAB else Path.cwd()
DATA_DIR = BASE_DIR / 'data' if (BASE_DIR / 'data').is_dir() else BASE_DIR
CODE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / 'a2_results'
STUDENT_ID = '415511001'  # 已依使用者提供的 Kaggle 截圖核對
STUDENT_NAME = '嚴佳華'  # 已依截圖核對；不會自動修改網站上的名稱
KAGGLE_TEAM_NAME = f'{STUDENT_ID}_{STUDENT_NAME}' if STUDENT_ID and STUDENT_NAME else 'StudentID_Name'
COLAB_URL = ''   # 填入可檢視的 Google Colab 連結
LEADERBOARD_RMSE = 12.37017  # 截圖中的實際成績；變更預測後須更新對應分數，未知時設 None
TA_BASELINE_RMSE = 24.2461  # 已核對正式 ZIP 的 baseline.txt: TA baseline test RMSE = 24.2461
RUN_OFFICIAL_EXPERIMENTS = True
QUICK_CHECK_ONLY = False  # 正式作業請保留 False
REUSE_SAVED_RESULTS = True  # 核對 CSV 與 pipeline SHA-256 後展示既有結果；完整重訓請設 False

print('Python:', sys.version.split()[0])
print('scikit-learn:', sklearn.__version__)
print('Random seed:', SEED)
print('Data directory:', DATA_DIR)
print('Result directory:', OUTPUT_DIR)
print('Kaggle Team Name:', KAGGLE_TEAM_NAME)
if LEADERBOARD_RMSE is not None:
    print(f'Recorded Kaggle test RMSE: {LEADERBOARD_RMSE:.5f}')
else:
    print('Kaggle test RMSE is pending an actual scored submission.')


Python: 3.13.18
scikit-learn: 1.6.1
Random seed: 0
Data directory: /content
Result directory: /content/a2_results
Kaggle Team Name: 415511001_嚴佳華
Recorded Kaggle test RMSE: 12.37017


## 0. 正式資料上傳與確認

下格預設不啟動互動上傳。在 Colab 需要上傳時，把 `UPLOAD_FILES` 改為 `True`。
請一併上傳 `train.csv`、`test.csv`、`sample_submission.csv`、`baseline.txt`。
只上傳老師提供的原始資料；不要用自己產生的測試資料代替作業資料。
配套 ZIP 已含四個正式檔案。預設先使用同層 `data/`，否則使用目前工作目錄
（Colab 為 `/content`）。程式只將三個 CSV 列為實驗必需檔案；若移動檔案時
漏掉 `baseline.txt`，不會阻擋訓練，仍應補回它以保留官方門檻的來源。


In [ ]:
UPLOAD_FILES = False
if IN_COLAB and UPLOAD_FILES:
    from google.colab import files
    uploaded = files.upload()
    for filename, content in uploaded.items():
        (DATA_DIR / Path(filename).name).write_bytes(content)

REQUIRED_CSV = ['train.csv', 'test.csv', 'sample_submission.csv']
missing_csv = [name for name in REQUIRED_CSV if not (DATA_DIR / name).is_file()]
DATA_READY = not missing_csv
if missing_csv:
    print('正式實驗尚未執行；缺少：' + ', '.join(missing_csv))
else:
    print('三個 CSV 已找到，執行時還會檢查欄位、月份、重複資料與 ID。')
print('依 2026-10-19 助教公告：starter 為選用，可繼續使用本 Notebook 的獨立實作。')


正式實驗尚未執行；缺少：train.csv, test.csv, sample_submission.csv
依 2026-10-19 助教公告：starter 為選用，可繼續使用本 Notebook 的獨立實作。


## 0.1 官方 baseline 參考與實算結果的區別

下格只顯示 `baseline.txt` 的原文，**不會自動解析數字或修改 `TA_BASELINE_RMSE`**。
正式資料包的原文已確認寫明 **TA baseline test RMSE = 24.2461**，因此設定格
已預填此值。這是官方 baseline 的成績，並非本次模型的 Kaggle 成績。
若改用其他課程資料，須重新確認其官方門檻，不能沿用不相符的分數。

本程式用 1–4 月訓練、5 月驗證所算出的 weather-only baseline，是本地比較結果；
**它不等於官方 Kaggle test baseline 分數，不能代替排行榜的門檻。**
排行榜項目的滿分條件是 **Kaggle test RMSE 嚴格小於官方 TA baseline RMSE**
（`LEADERBOARD_RMSE < TA_BASELINE_RMSE`）；相等不算達標。
最終比較必須使用實際 Kaggle 分數與官方對應 baseline；本地 5 月驗證分數
無法證明已達到這項排行榜要求。

原文另列 train RMSE 25.0947、train R² 0.1437、May RMSE 29.3875、May R² 0.1437。
下方會並列本次實算數值；若 R² 與原文不一致，保留實算值並說明差異，
不將結果改成文件中的數字。


In [ ]:
baseline_path = DATA_DIR / 'baseline.txt'
BASELINE_REFERENCE_AVAILABLE = baseline_path.is_file()
baseline_reference_text = None
if BASELINE_REFERENCE_AVAILABLE:
    try:
        baseline_reference_text = baseline_path.read_text(encoding='utf-8-sig')
    except (OSError, UnicodeError) as exc:
        print(f'baseline.txt 尚無法讀取（{type(exc).__name__}）；請確認文字編碼或直接閱讀原檔。')
        print('不影響三個 CSV 的訓練與預測；請保留正式來源供核對，已知門檻為 24.2461。')
    else:
        print('官方 baseline.txt 原文：')
        print(baseline_reference_text)
        if not baseline_reference_text.strip():
            print('baseline.txt 內容為空；請補上官方參考內容。')
        print('設定格的 24.2461 已由此正式檔核對；本格沒有自動擷取或覆寫任何分數。')
else:
    print('此工作目錄缺少 baseline.txt；訓練與預測可繼續，請補回正式參考檔。')
    print('已核對的官方門檻為 24.2461；請保留 baseline.txt 作為門檻來源。')
print('本地 5 月 weather-only baseline 不能代替官方 Kaggle test baseline。')
print('目前 TA_BASELINE_RMSE：', TA_BASELINE_RMSE)


此工作目錄缺少 baseline.txt；訓練與預測可繼續，請補回正式參考檔。
已核對的官方門檻為 24.2461；請保留 baseline.txt 作為門檻來源。
本地 5 月 weather-only baseline 不能代替官方 Kaggle test baseline。
目前 TA_BASELINE_RMSE： 24.2461


## 0.2 官方資料欄位與提交格式

依使用者提供的 Kaggle 競賽頁面與已收到的正式資料，欄位如下。
程式仍會核對實際檔案的欄位、月份與 ID，並保留檔案 SHA-256。

| 欄位 | 意義／單位 |
|---|---|
| `id` | 每筆觀測資料的識別碼 |
| `datetime` | 觀測的小時時間戳記 |
| `station` | YouBike 站點 |
| `district` | 所在行政區 |
| `capacity` | 站點的停車柱／泊位數（docks） |
| `is_holiday` | 假日指標 |
| `temperature` | 溫度，°C |
| `precipitation` | 降雨量，mm |
| `humidity` | 相對濕度，% |
| `wind_speed` | 風速，km/h |
| `rentals` | 該站每小時的出發租借次數；訓練資料的預測目標 |

`capacity` 是站點的泊位數，`rentals` 是一小時累計的出發次數，
因此 **租借次數不以 capacity 為上限**。預測僅將負值裁切為 0，
不以站點容量封頂，也不強制四捨五入。

**競賽提交頁的範例欄頭是 `ID,Rentals`，這次收到的實際範本欄頭是
`id,rentals`。`sample_submission.csv` 才是提交格式的依據。** 程式會依該範本的 ID 對齊
預測，保留範本的列順序、欄位順序和欄名大小寫；不會強制改成 `id,rentals`。
資料字典中的小寫欄名不代表提交範本也必須使用小寫。


# Part I. Least Squares by Hand

The handout (p. 1) permits **handwritten or typeset solutions**. The typeset
derivations below satisfy the same calculation and explanation requirements.

## (a) Ridge solution and uniqueness

For the stated objective
\[
J(w)=\frac12\lVert y-Xw\rVert_2^2+\frac\lambda2\lVert w\rVert_2^2,
\]
the gradient is
\[
\nabla J(w)=X^\top(Xw-y)+\lambda w.
\]
Setting it to zero gives
\[
(X^\top X+\lambda I)w=X^\top y,
\qquad w^*=(X^\top X+\lambda I)^{-1}X^\top y.
\]
For any nonzero vector $v$ and $\lambda>0$,
\[
v^\top(X^\top X+\lambda I)v=\lVert Xv\rVert_2^2+
\lambda\lVert v\rVert_2^2>0.
\]
Thus the Hessian is positive definite and the solution is unique, even when
$X^\top X$ is singular.

**Unpenalized intercept.** The toy-data task leaves the intercept unpenalized.
If a constant column is included in $X$, the penalty matrix is
$P=\mathrm{diag}(0,1,\ldots,1)$ rather than $I$. Equivalently, center the
features and target, solve for the slopes, and recover the intercept.

For $x=(0,1,2,3,4)$ and $y=(10,8,7,4,1)$:
\[
\bar x=2,\quad\bar y=6,\quad S_{xx}=10,\quad S_{xy}=-22,\quad S_{yy}=50.
\]
The OLS coefficients are
\[
w_1=-22/10=-2.2,\qquad w_0=6-(-2.2)(2)=10.4.
\]
The predictions are $(10.4,8.2,6,3.8,1.6)$, so
\[
SSE=1.6,\qquad MSE=1.6/5=0.32,\qquad R^2=1-1.6/50=0.968.
\]
The MSE denominator is $N$, not $N-2$.

With an unpenalized intercept, Ridge uses
\[
w_1(\lambda)=\frac{-22}{10+\lambda},\qquad w_0=6-2w_1.
\]
At $\lambda=1$, $(w_0,w_1)=(10,-2)$; at $\lambda=10$, they are $(8.2,-1.1)$.
As $\lambda\to\infty$, the slope tends to zero and the intercept tends to
$\bar y=6$.


In [ ]:
# 真正驗算題目中的五筆資料；這不是 Part II–III 的實驗結果。
x = np.array([0., 1., 2., 3., 4.])
y = np.array([10., 8., 7., 4., 1.])
xbar, ybar = x.mean(), y.mean()
Sxx = np.sum((x-xbar)**2)
Sxy = np.sum((x-xbar)*(y-ybar))
Syy = np.sum((y-ybar)**2)
toy_rows = []
for lam in [0., 1., 10.]:
    slope = Sxy / (Sxx+lam)
    intercept = ybar-slope*xbar
    predictions = intercept+slope*x
    sse = np.sum((y-predictions)**2)
    toy_rows.append({'model': 'OLS' if lam == 0 else f'Ridge lambda={lam:g}',
                     'intercept': intercept, 'slope_per_mm': slope,
                     'MSE': sse/len(y), 'R2': 1-sse/Syy})
print(f'x_mean={xbar:g}, y_mean={ybar:g}, Sxx={Sxx:g}, Sxy={Sxy:g}, Syy={Syy:g}')
print(pd.DataFrame(toy_rows).to_string(index=False))
print('OLS predictions:', (10.4-2.2*x).tolist())


x_mean=2, y_mean=6, Sxx=10, Sxy=-22, Syy=50
          model  intercept  slope_per_mm  MSE    R2
            OLS       10.4          -2.2 0.32 0.968
 Ridge lambda=1       10.0          -2.0 0.40 0.960
Ridge lambda=10        8.2          -1.1 2.74 0.726
OLS predictions: [10.4, 8.2, 6.0, 3.8, 1.5999999999999996]


## (b) Changing the rain gauge from millimetres to centimetres

Let $z=x/10$. Then $\bar z=0.2$, $S_{zz}=0.1$, and $S_{zy}=-2.2$.
OLS gives a slope of $-22$ rentals per centimetre and an intercept of $10.4$.
Because $10.4-22z=10.4-2.2x$, its predictions, MSE, and $R^2$ do not change.

For Ridge with $\lambda=1$:
\[
w_{1,\mathrm{cm}}=\frac{-2.2}{0.1+1}=-2,
\qquad w_0=6-(-2)(0.2)=6.4.
\]
The effect per millimetre is $-2/10=-0.2$, compared with $-2$ per millimetre
for the original Ridge model. Its predictions are $(6.4,6.2,6,5.8,5.6)$.

To represent the same function after dividing an input by 10, its coefficient
must become 10 times larger, giving a 100-fold squared penalty at fixed
$\lambda$. For this one-feature example, preserving the original fit requires
$\lambda_{\mathrm{cm}}=\lambda_{\mathrm{mm}}/100$.

**Gradient descent.** Widely different feature scales can produce a poorly
conditioned quadratic loss. A large step size may oscillate or diverge along
steep directions; a small safe step may progress slowly along flatter ones.
For a positive-definite quadratic with Hessian $H$, full-batch gradient descent
with a fixed step converges when $0<\eta<2/\lambda_{\max}(H)$.
Scaling can improve conditioning, but unscaled data do not necessarily diverge,
and scaling does not remove all feature correlations.


In [ ]:
z = x/10
Szz = np.sum((z-z.mean())**2)
Szy = np.sum((z-z.mean())*(y-y.mean()))
unit_rows = []
for method, lam in [('OLS in cm', 0.), ('Ridge in cm, lambda=1', 1.),
                     ('Ridge in cm, lambda=0.01', 0.01)]:
    slope_cm = Szy/(Szz+lam)
    b = y.mean()-slope_cm*z.mean()
    unit_rows.append({'model': method, 'intercept': b,
                      'slope_per_cm': slope_cm, 'effect_per_mm': slope_cm/10})
print(pd.DataFrame(unit_rows).to_string(index=False))
assert np.allclose(10.4-22*z, 10.4-2.2*x)
print('Verified: OLS predictions are unchanged by the unit conversion.')


                   model  intercept  slope_per_cm  effect_per_mm
               OLS in cm       10.4         -22.0           -2.2
   Ridge in cm, lambda=1        6.4          -2.0           -0.2
Ridge in cm, lambda=0.01       10.0         -20.0           -2.0
Verified: OLS predictions are unchanged by the unit conversion.


# Part II–III. Reproducible implementation

The next two cells create the complete analysis and PDF-report modules locally.
All trainable preprocessing is fitted inside each pipeline. Formal model
selection uses May only; the final model is refitted on January–May.

**Feature experiment (F0–F4):** weather → time → station → hour interactions →
station/hour/weekday interactions, with a fixed Ridge configuration. **Scaling experiment:** three numeric
preprocessors × three models. **Learning curves:** full-timestamp forward
splits, with fresh pipelines at every training size. The TA announcement of
19 October 2026 permits an existing independent implementation. The course
helper can optionally be connected; its absence does not block this workflow.

Final model selection first screens the eligible representations using
January-April temporal-CV stability evidence, then minimizes clipped May RMSE
within that eligible set. The original degree-4 weather model B is retained as
the high-variance diagnostic example. Its remedy reduces weather polynomial
degree, fits weather-range bounds only on each training fold, and applies Ridge.
The recorded selection policy and development note identify any exclusions.


In [ ]:
%%writefile a2_pipeline.py
"""A2: The Morning Rush — 可重現的線性迴歸實驗。

正式資料必須由課程提供。此程式不下載替代資料，也不製造作業分數。
資料夾須含 train.csv、test.csv、sample_submission.csv。
依助教 2026-10-19 公告，可繼續使用自行撰寫的程式；starter notebook
與提供的資料讀取、切分、繪圖 helper 均為參考。本模組以自行實作的
完整小時時間切分學習曲線為主；若另傳入老師 helper，會如實記錄呼叫結果。

用法：
    python a2_pipeline.py --data-dir . --output-dir a2_results --student-id YOUR_ID
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import inspect
import json
import math
import os
import random
import tempfile
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

# Colab 與唯讀家目錄的執行環境皆使用可寫的 Matplotlib 快取位置。
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "a2_matplotlib"))
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, SGDRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, PolynomialFeatures, StandardScaler

SEED = 0
WEATHER = ["temperature", "precipitation", "humidity", "wind_speed"]
REQUIRED_X = ["id", "datetime", "station", "district", "capacity", "is_holiday"] + WEATHER
FEATURE_DESCRIPTIONS = {
    "F0": "Four weather measurements only.",
    "F1": "F0 + hour and weekday one-hot encodings + holiday indicator.",
    "F2": "F1 + station one-hot encoding.",
    "F3": "F2 + station-by-hour and holiday-by-hour categorical interactions.",
    "F4": "F3 + station-by-hour-by-weekday interactions; weather terms remain linear.",
    "B": "F4 + unbounded degree-4 weather polynomial terms; deliberately unstable diagnostic model, not eligible for final selection.",
    "B_stable": "F4 + degree-2 weather terms after clipping each weather input to bounds learned only from the current training partition.",
}


def _one_hot():
    # 新舊 Colab sklearn 都可用；每次只在訓練集 fit。
    options = dict(handle_unknown="ignore", drop="first", dtype=np.float64)
    try:
        return OneHotEncoder(sparse_output=True, **options)
    except TypeError:
        return OneHotEncoder(sparse=True, **options)


class CalendarFeatures(BaseEstimator, TransformerMixin):
    """只依輸入欄位做固定規則轉換，不學習目標或全資料統計量。"""

    def fit(self, X, y=None):
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("CalendarFeatures requires a pandas DataFrame.")
        out = X.copy()
        dt = pd.to_datetime(out["datetime"], errors="raise")
        out["hour"] = dt.dt.hour.astype(str)
        out["weekday"] = dt.dt.dayofweek.astype(str)
        out["station"] = out["station"].astype(str)
        out["is_holiday"] = pd.to_numeric(out["is_holiday"], errors="raise").astype(float)
        # JSON 字串編碼 tuple，避免站名本身含分隔符號造成鍵碰撞。
        out["station_hour"] = [json.dumps([s, h], ensure_ascii=False) for s, h in zip(out["station"], out["hour"])]
        out["holiday_hour"] = [json.dumps([int(d), h]) for d, h in zip(out["is_holiday"], out["hour"])]
        out["station_hour_weekday"] = [json.dumps([s, h, d], ensure_ascii=False) for s, h, d in zip(out["station"], out["hour"], out["weekday"])]
        return out


class TrainingRangeClipper(BaseEstimator, TransformerMixin):
    """只從 fit 輸入學習每個天氣欄位的範圍，限制多項式在範圍外外推。

    此轉換器置於 Pipeline 的補值之後、縮放及多項式之前。CV 每折及每個
    訓練大小都重新 fit；最終重訓僅使用 Jan-May 更新範圍，不讀 June 資料。
    這是對輸入天氣做限制，不是限制預測租借量，也不是用車柱容量截上限。
    """

    def fit(self, X, y=None):
        values = np.asarray(X, dtype=float)
        if values.ndim != 2 or values.shape[0] == 0 or not np.isfinite(values).all():
            raise ValueError("TrainingRangeClipper requires a non-empty finite 2D training array after imputation.")
        self.n_features_in_ = values.shape[1]
        self.data_min_ = np.min(values, axis=0)
        self.data_max_ = np.max(values, axis=0)
        return self

    def transform(self, X):
        if not hasattr(self, "data_min_"):
            raise ValueError("TrainingRangeClipper must be fitted on training data first.")
        values = np.asarray(X, dtype=float)
        if values.ndim != 2 or values.shape[1] != self.n_features_in_ or not np.isfinite(values).all():
            raise ValueError("TrainingRangeClipper received non-finite values or a mismatched feature shape.")
        return np.clip(values, self.data_min_, self.data_max_)


def make_estimator(feature_set="F3", model="Ridge", alpha=10.0, scaler="standard", target="raw", quick=False):
    """每次傳回尚未 fit 的完整 Pipeline（或包住它的 TTR）。"""
    if feature_set not in FEATURE_DESCRIPTIONS:
        raise ValueError(f"Unknown feature set: {feature_set}")
    if scaler not in {"none", "standard", "minmax"}:
        raise ValueError(f"Unknown scaler: {scaler}")
    if target not in {"raw", "log1p"}:
        raise ValueError(f"Unknown target transform: {target}")
    scale = {"none": "passthrough", "standard": StandardScaler(), "minmax": MinMaxScaler()}[scaler]
    numeric_steps = [("impute", SimpleImputer(strategy="median"))]
    if feature_set == "B_stable":
        numeric_steps.append(("weather_bounds", TrainingRangeClipper()))
    numeric_steps.append(("scale", scale))
    if feature_set in {"B", "B_stable"}:
        # 高階展開只作用於四個天氣欄位，不作用於上千個 one-hot 欄位。
        degree = 4 if feature_set == "B" else 2
        numeric_steps += [("poly", PolynomialFeatures(degree=degree, include_bias=False)), ("poly_scale", StandardScaler())]
    blocks = [("numeric", Pipeline(numeric_steps), WEATHER)]
    cats = []
    if feature_set != "F0":
        cats += ["hour", "weekday"]
        blocks.append(("holiday", "passthrough", ["is_holiday"]))
    if feature_set in {"F2", "F3", "F4", "B", "B_stable"}:
        cats += ["station"]
    if feature_set in {"F3", "F4", "B", "B_stable"}:
        cats += ["station_hour", "holiday_hour"]
    if feature_set in {"F4", "B", "B_stable"}:
        cats += ["station_hour_weekday"]
    if cats:
        blocks.append(("categorical", _one_hot(), cats))
    preprocess = ColumnTransformer(blocks, remainder="drop", sparse_threshold=1.0)
    if model == "LinearRegression":
        reg = LinearRegression()
    elif model == "Ridge":
        # 顯式使用 lsqr，避免舊 sklearn / scipy sparse_cg 介面差異。
        reg = Ridge(alpha=float(alpha), solver="lsqr", tol=1e-6)
    elif model == "SGDRegressor":
        reg = SGDRegressor(
            loss="squared_error", penalty=None, learning_rate="invscaling",
            eta0=0.01, power_t=0.25, max_iter=150 if quick else 2000,
            tol=1e-3, n_iter_no_change=10, early_stopping=False,
            random_state=SEED, shuffle=True,
        )
    else:
        raise ValueError(f"Model is not permitted by this assignment: {model}")
    pipeline = Pipeline([("calendar", CalendarFeatures()), ("preprocess", preprocess), ("model", reg)])
    if target == "log1p":
        return TransformedTargetRegressor(regressor=pipeline, func=np.log1p, inverse_func=np.expm1, check_inverse=True)
    return pipeline


def _pipeline(estimator):
    return estimator.regressor_ if isinstance(estimator, TransformedTargetRegressor) else estimator


def learned_weather_bounds(estimator):
    """回傳已 fit 模型實際學到的天氣界限，供報告追蹤；沒有裁切則回傳 None。"""
    numeric = _pipeline(estimator).named_steps["preprocess"].named_transformers_["numeric"]
    bounder = numeric.named_steps.get("weather_bounds")
    if bounder is None:
        return None
    return {name: {"min": float(lower), "max": float(upper)} for name, lower, upper in zip(WEATHER, bounder.data_min_, bounder.data_max_)}


def _json_safe(value):
    if isinstance(value, dict):
        return {str(k): _json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, np.ndarray)):
        return [_json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (float, np.floating)):
        return float(value) if np.isfinite(value) else None
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    return value


def _write_json(path, value):
    Path(path).write_text(json.dumps(_json_safe(value), ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")


def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def _submission_columns(columns):
    """辨識模板的 ID 欄；只比較大小寫，保留實際欄名及原始欄序。"""
    columns = list(columns)
    if len(columns) != 2 or any(not isinstance(col, str) or not col.strip() for col in columns):
        raise ValueError("sample_submission.csv must contain exactly two non-empty column names: one ID column and one prediction column.")
    if len(set(columns)) != len(columns):
        raise ValueError("sample_submission.csv contains duplicate column names.")
    id_columns = [col for col in columns if col.casefold() == "id"]
    if len(id_columns) != 1:
        raise ValueError("sample_submission.csv must contain exactly one ID column (case-insensitive); missing or ambiguous ID headers are invalid.")
    id_column = id_columns[0]
    prediction_column = next(col for col in columns if col != id_column)
    return id_column, prediction_column


def _validate_ids(ids, source):
    # ID 是識別字串；保留前導零、大小寫與 NA 等文字，不轉成數值。
    ids = ids.astype("string")
    if ids.isna().any() or ids.eq("").any() or ids.duplicated().any():
        raise ValueError(f"{source} ids must be unique and non-empty.")
    return ids


def _validate_submission_alignment(test, sample):
    id_column, prediction_column = _submission_columns(sample.columns)
    if "id" not in test.columns:
        raise ValueError("test.csv must contain the lowercase id column specified by the dataset schema.")
    test_ids = _validate_ids(test["id"], "test.csv")
    sample_ids = _validate_ids(sample[id_column], "sample_submission.csv")
    if len(sample) != len(test) or set(sample_ids) != set(test_ids):
        raise ValueError("test.csv and sample_submission.csv id sets must match exactly.")
    return id_column, prediction_column, test_ids, sample_ids


def construct_submission(test, sample, predictions):
    """使用真正輸出路徑的對齊規則，精確保留模板欄名、欄序與 ID 順序。"""
    id_column, prediction_column, test_ids, sample_ids = _validate_submission_alignment(test, sample)
    predictions = np.asarray(predictions, dtype=float).reshape(-1)
    if len(predictions) != len(test) or not np.isfinite(predictions).all():
        raise ValueError("Final test predictions are non-finite or have the wrong number of rows.")
    by_id = pd.Series(np.maximum(predictions, 0.0), index=test_ids)
    aligned = sample_ids.map(by_id)
    if aligned.isna().any() or not np.isfinite(aligned.to_numpy(dtype=float)).all():
        raise AssertionError("Submission alignment failed.")
    submission = sample.copy()
    submission[prediction_column] = aligned.to_numpy(dtype=float)
    if list(submission.columns) != list(sample.columns) or not submission[id_column].equals(sample[id_column]):
        raise AssertionError("Submission does not preserve the sample schema and id order.")
    return submission


def load_assignment_data(data_dir):
    folder = Path(data_dir)
    required = ["train.csv", "test.csv", "sample_submission.csv"]
    missing = [name for name in required if not (folder / name).is_file()]
    if missing:
        raise FileNotFoundError(
            "缺少正式課程資料：" + ", ".join(missing)
            + "。請從 e3 取得後放入資料夾；本程式不會製造資料或模型分數。"
        )
    dtype = {"station": "string", "district": "string"}
    # converters 保留原始 ID 字串（含前導零及文字 NA）；其他欄位沿用正常缺值解析。
    train = pd.read_csv(folder / "train.csv", dtype=dtype, converters={"id": str})
    test = pd.read_csv(folder / "test.csv", dtype=dtype, converters={"id": str})
    # 先檢查原始標頭，避免 pandas 將重複標頭改成 id.1 而掩蓋無效模板。
    with (folder / "sample_submission.csv").open("r", encoding="utf-8-sig", newline="") as handle:
        sample_headers = next(csv.reader(handle), [])
    _submission_columns(sample_headers)
    # 模板首次讀取即保留字串，不能等 pandas 把 001 讀成 1 後再轉字串。
    sample = pd.read_csv(folder / "sample_submission.csv", dtype="string", keep_default_na=False)
    checks = []
    for name, frame in [("train", train), ("test", test)]:
        missing_cols = sorted(set(REQUIRED_X + (["rentals"] if name == "train" else [])) - set(frame.columns))
        if missing_cols:
            raise ValueError(f"{name}.csv 缺少欄位：{missing_cols}")
        if frame.empty:
            raise ValueError(f"{name}.csv is empty.")
        frame["id"] = _validate_ids(frame["id"], f"{name}.csv")
        if frame["station"].isna().any():
            raise ValueError(f"{name}.csv contains a missing station.")
        frame["datetime"] = pd.to_datetime(frame["datetime"], errors="raise")
        if frame["datetime"].isna().any():
            raise ValueError(f"{name}.csv contains a missing datetime.")
        if frame["datetime"].dt.tz is not None:
            raise ValueError("資料日期應為講義的當地小時，請先確認時區，不自動換日。")
        if not (frame["datetime"] == frame["datetime"].dt.floor("h")).all():
            raise ValueError(f"{name}.csv datetime must be the start of each hour.")
        if frame.duplicated(["station", "datetime"]).any():
            raise ValueError(f"{name}.csv has duplicate (station, datetime) rows.")
        for col in WEATHER + ["capacity"]:
            frame[col] = pd.to_numeric(frame[col], errors="raise")
            observed = frame[col].dropna().to_numpy(dtype=float)
            if not np.isfinite(observed).all():
                raise ValueError(f"{name}.csv {col} contains infinite values.")
        holiday = frame["is_holiday"].replace({True: 1, False: 0, "True": 1, "False": 0, "true": 1, "false": 0})
        frame["is_holiday"] = pd.to_numeric(holiday, errors="raise")
        if frame["is_holiday"].isna().any() or not frame["is_holiday"].isin([0, 1]).all():
            raise ValueError(f"{name}.csv is_holiday must contain only 0 or 1.")
        if (frame["precipitation"].dropna() < 0).any():
            raise ValueError(f"{name}.csv contains negative rainfall.")
    expected_train = set(pd.period_range("2026-01", "2026-05", freq="M"))
    observed_train = set(train["datetime"].dt.to_period("M"))
    observed_test = set(test["datetime"].dt.to_period("M"))
    if observed_train != expected_train:
        raise ValueError(f"train.csv must cover Jan-May 2026 only; observed months: {sorted(map(str, observed_train))}")
    if observed_test != {pd.Period("2026-06", freq="M")}:
        raise ValueError("test.csv must contain June 2026 only.")
    train["rentals"] = pd.to_numeric(train["rentals"], errors="raise")
    if not np.isfinite(train["rentals"].to_numpy(dtype=float)).all() or (train["rentals"] < 0).any():
        raise ValueError("Training rentals must be finite and non-negative.")
    if "rentals" in test.columns:
        checks.append("A rentals column exists in test.csv; it is ignored and never used for evaluation or model selection.")
    sample_id_col, prediction_col, _, _ = _validate_submission_alignment(test, sample)
    for name, frame in [("train", train), ("test", test)]:
        stations = frame["station"].nunique()
        hours = frame["datetime"].nunique()
        if stations != 30:
            checks.append(f"{name}.csv has {stations} stations; the handout describes 30. Verify that these are the official complete files.")
        absent_pairs = stations * hours - len(frame)
        if absent_pairs:
            checks.append(f"{name}.csv is missing {absent_pairs} station-hour pairs within its observed hour grid. No targets are filled or invented.")
        expected_hours = len(pd.date_range("2026-01-01", "2026-06-01", freq="h", inclusive="left")) if name == "train" else 30 * 24
        if hours != expected_hours:
            checks.append(f"{name}.csv has {hours} unique hours, versus {expected_hours} for the full stated date range. Verify file completeness.")
    unseen = sorted(set(test["station"]) - set(train["station"]))
    if unseen:
        checks.append(f"Test includes {len(unseen)} stations absent from training; unknown categories are encoded as zeros.")
    # 按日期排序後才交叉驗證；提交時另以 id 對齊回 sample_submission。
    train = train.sort_values(["datetime", "station"], kind="stable").reset_index(drop=True)
    test = test.reset_index(drop=True)
    fit = train.loc[train["datetime"] < pd.Timestamp("2026-05-01")].reset_index(drop=True)
    validation = train.loc[train["datetime"] >= pd.Timestamp("2026-05-01")].reset_index(drop=True)
    if fit.empty or validation.empty:
        raise ValueError("The Jan-Apr training split and May validation split must both be non-empty.")
    if fit[WEATHER].isna().all().any():
        raise ValueError("A weather feature is entirely missing in the Jan-Apr training split.")
    summary = {
        "train_rows": len(train), "fit_rows": len(fit), "validation_rows": len(validation), "test_rows": len(test),
        "station_count": int(train["station"].nunique()), "submission_id_column": sample_id_col, "prediction_column": prediction_col,
        "split_dates": {"fit": "2026-01-01 to 2026-04-30", "validation": "2026-05-01 to 2026-05-31", "final_fit": "2026-01-01 to 2026-05-31", "test": "2026-06-01 to 2026-06-30"},
        "missing_values": {"train": train[REQUIRED_X + ["rentals"]].isna().sum().to_dict(), "test": test[REQUIRED_X].isna().sum().to_dict()},
        "checks": checks,
        "input_sha256": {name: _sha256(folder / name) for name in required},
    }
    return train, fit, validation, test, sample, summary


def regression_metrics(y, prediction):
    y = np.asarray(y, dtype=float)
    prediction = np.asarray(prediction, dtype=float).reshape(-1)
    if len(y) != len(prediction) or not np.isfinite(prediction).all():
        raise ValueError("Predictions are non-finite or have the wrong length.")
    rmse = float(np.sqrt(mean_squared_error(y, prediction)))
    r2 = float(r2_score(y, prediction))
    if not np.isfinite(rmse) or not np.isfinite(r2):
        raise ValueError("The calculated RMSE or R2 is non-finite.")
    return rmse, r2


def fit_and_evaluate(estimator, X_train, y_train, X_val, y_val):
    started = time.perf_counter()
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always")
        estimator.fit(X_train, y_train)
        pred_train = estimator.predict(X_train)
        pred_val = estimator.predict(X_val)
        train_rmse, train_r2 = regression_metrics(y_train, pred_train)
        val_rmse, val_r2 = regression_metrics(y_val, pred_val)
        clip_rmse, clip_r2 = regression_metrics(y_val, np.maximum(pred_val, 0.0))
    pipe = _pipeline(estimator)
    n_features = int(pipe[:-1].transform(X_train.iloc[:1]).shape[1])
    reg = pipe.named_steps["model"]
    messages = list(dict.fromkeys(f"{type(w.message).__name__}: {w.message}" for w in captured))
    row = {
        "n_features": n_features, "train_rmse": train_rmse, "train_r2": train_r2,
        "val_rmse": val_rmse, "val_r2": val_r2,
        "val_rmse_clipped": clip_rmse, "val_r2_clipped": clip_r2,
        "negative_prediction_fraction": float(np.mean(np.asarray(pred_val) < 0)),
        "epochs": int(reg.n_iter_) if isinstance(reg, SGDRegressor) else None,
        "status": "warning" if messages else "ok", "warnings": messages,
        "fit_seconds": time.perf_counter() - started,
        "weather_bounds": learned_weather_bounds(estimator),
    }
    return row, estimator


def _savefig(fig, path):
    fig.savefig(path, dpi=160, bbox_inches="tight", facecolor="white")
    plt.close(fig)


def make_eda(fit, output_dir):
    """所有 EDA 都只使用 Jan-Apr，避免先偷看 May 來描述訓練模式。"""
    out = Path(output_dir)
    figures = {}
    data = fit.copy()
    data["hour"] = data["datetime"].dt.hour
    data["date"] = data["datetime"].dt.date
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.9))
    hourly = data.groupby(["is_holiday", "hour"])["rentals"].mean().unstack(0)
    for h in hourly.columns:
        axes[0].plot(hourly.index, hourly[h], marker="o", markersize=3, label="Holiday" if h else "Non-holiday")
    axes[0].set(xlabel="Hour", ylabel="Mean rentals per station-hour", title="Hourly pattern (Jan-Apr)", xticks=range(0, 24, 3))
    axes[0].legend()
    daily = data.groupby("date")["rentals"].mean()
    axes[1].plot(pd.to_datetime(daily.index), daily.values, linewidth=1.1)
    axes[1].set(xlabel="Date", ylabel="Mean rentals per station-hour", title="Daily demand (Jan-Apr)")
    axes[1].tick_params(axis="x", rotation=25)
    fig.tight_layout()
    _savefig(fig, out / "figures/eda_hourly.png")
    figures["eda_hourly"] = "figures/eda_hourly.png"

    station_means = data.groupby("station")["rentals"].mean().sort_values(ascending=False)
    station_table = pd.DataFrame({"station_rank": np.arange(1, len(station_means) + 1), "station": station_means.index, "mean_rentals": station_means.values})
    station_table.to_csv(out / "tables/station_summary.csv", index=False)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), gridspec_kw={"width_ratios": [1, 1.25]})
    axes[0].bar(np.arange(1, len(station_means) + 1), station_means.values, color="#247f88")
    axes[0].set(xlabel="Station rank (mapping in station_summary.csv)", ylabel="Mean rentals", title="Demand differs by station")
    heat = data.groupby(["station", "hour"])["rentals"].mean().unstack("hour").reindex(station_means.index)
    im = axes[1].imshow(heat, aspect="auto", cmap="YlGnBu", origin="upper")
    axes[1].set(xlabel="Hour", ylabel="Station rank", title="Station-by-hour mean demand")
    axes[1].set_xticks(np.arange(len(heat.columns))[::3], labels=heat.columns[::3])
    axes[1].set_yticks(np.arange(len(heat.index))[::max(1, len(heat.index) // 8)], labels=np.arange(1, len(heat.index) + 1)[::max(1, len(heat.index) // 8)])
    fig.colorbar(im, ax=axes[1], label="Mean rentals")
    fig.tight_layout()
    _savefig(fig, out / "figures/eda_stations.png")
    figures["eda_stations"] = "figures/eda_stations.png"

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.9))
    rain_group = pd.cut(data["precipitation"], [-1e-12, 0, 0.5, 2, 5, 10, np.inf], labels=["0", "0-0.5", "0.5-2", "2-5", "5-10", ">10"])
    rain_stats = data.groupby(rain_group, observed=True)["rentals"].agg(["mean", "count"])
    axes[0].bar(rain_stats.index.astype(str), rain_stats["mean"], color="#487ca3")
    axes[0].set(xlabel="Precipitation (mm)", ylabel="Mean rentals", title="Rainfall groups (association only)")
    try:
        temp_group = pd.qcut(data["temperature"], q=10, duplicates="drop")
        temp_stats = data.groupby(temp_group, observed=True).agg(temperature=("temperature", "mean"), rentals=("rentals", "mean"), count=("rentals", "size"))
        if temp_stats.empty:
            raise ValueError("Constant or missing temperature")
        axes[1].plot(temp_stats["temperature"], temp_stats["rentals"], marker="o")
    except ValueError:
        axes[1].text(0.5, 0.5, "Insufficient temperature variation", ha="center", transform=axes[1].transAxes)
    axes[1].set(xlabel="Mean temperature in quantile bin (C)", ylabel="Mean rentals", title="Temperature groups (Jan-Apr)")
    fig.tight_layout()
    _savefig(fig, out / "figures/eda_weather.png")
    figures["eda_weather"] = "figures/eda_weather.png"

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.9))
    axes[0].hist(data["rentals"], bins=50, color="#247f88")
    axes[0].set(xlabel="Rentals", ylabel="Number of station-hours", title="Raw target distribution")
    axes[1].hist(np.log1p(data["rentals"]), bins=50, color="#ab714b")
    axes[1].set(xlabel="log(1 + rentals)", ylabel="Number of station-hours", title="Log-transformed target distribution")
    fig.tight_layout()
    _savefig(fig, out / "figures/eda_target.png")
    figures["eda_target"] = "figures/eda_target.png"
    hourly_overall = data.groupby("hour")["rentals"].mean()
    desc = {
        "source_split": "January-April only", "mean_rentals": float(data["rentals"].mean()),
        "median_rentals": float(data["rentals"].median()), "zero_fraction": float((data["rentals"] == 0).mean()),
        "max_rentals": float(data["rentals"].max()), "target_skewness": float(data["rentals"].skew()),
        "peak_hour_overall": int(hourly_overall.idxmax()), "lowest_hour_overall": int(hourly_overall.idxmin()),
        "highest_station_mean": float(station_means.max()), "lowest_station_mean": float(station_means.min()),
        "rain_groups": [{"rain_mm": str(k), "mean_rentals": float(v["mean"]), "count": int(v["count"])} for k, v in rain_stats.iterrows()],
        "interpretation_note": "These are descriptive associations, not causal effects. Hours and station identity are motivated by the observed group differences; feature ablations determine predictive value.",
    }
    return desc, figures


def make_temporal_folds(frame, n_splits=3):
    """依唯一 datetime 切分；同一小時的所有站點一定留在同一側。"""
    hours = pd.DatetimeIndex(sorted(frame["datetime"].unique()))
    if len(hours) < 12:
        raise ValueError("At least 12 distinct Jan-Apr hours are required for temporal learning curves.")
    test_size = max(2, min(14 * 24, len(hours) // 8))
    splitter = TimeSeriesSplit(n_splits=n_splits, test_size=test_size)
    folds = []
    details = []
    for fold_id, (tr, va) in enumerate(splitter.split(hours)):
        train_hours, val_hours = hours[tr], hours[va]
        train_index = np.flatnonzero(frame["datetime"].isin(train_hours).to_numpy())
        val_index = np.flatnonzero(frame["datetime"].isin(val_hours).to_numpy())
        if frame.iloc[train_index]["datetime"].max() >= frame.iloc[val_index]["datetime"].min():
            raise AssertionError("A temporal fold crossed its validation boundary.")
        folds.append((train_index, val_index, train_hours))
        details.append({"fold": fold_id, "training_start": str(train_hours.min()), "training_end": str(train_hours.max()), "validation_start": str(val_hours.min()), "validation_end": str(val_hours.max()), "train_hours": len(train_hours), "val_hours": len(val_hours), "train_rows": len(train_index), "val_rows": len(val_index)})
    return folds, details


def compute_learning_curve(estimator, X, y, folds, quick=False):
    """整小時增加訓練資料，每個 fold/size 都 clone 並重 fit 全 Pipeline。"""
    common_hours = min(len(hours) for _, _, hours in folds)
    fractions = [0.15, 0.50, 1.0] if quick else [0.10, 0.25, 0.50, 0.75, 1.0]
    sizes = sorted(set(max(2, int(common_hours * f)) for f in fractions))
    raw_rows = []
    points = []
    for hour_count in sizes:
        outcomes = []
        for fold_id, (_, val_index, train_hours) in enumerate(folds):
            # 保持訓練結束點不變，向過去延長視窗；不把預測距離當成樣本量效應。
            selected_hours = train_hours[-hour_count:]
            train_index = np.flatnonzero(X["datetime"].isin(selected_hours).to_numpy())
            model = clone(estimator)
            try:
                row, _ = fit_and_evaluate(model, X.iloc[train_index], y.iloc[train_index], X.iloc[val_index], y.iloc[val_index])
                item = {"fold": fold_id, "train_hours": hour_count, "train_size": len(train_index), "train_rmse": row["train_rmse"], "validation_rmse": row["val_rmse"], "status": row["status"], "warnings": row["warnings"]}
                outcomes.append(item)
                raw_rows.append(item)
            except Exception as exc:
                raise RuntimeError(f"Learning curve failed at fold {fold_id}, {hour_count} training hours: {type(exc).__name__}: {exc}") from exc
        points.append({
            "train_hours": hour_count, "train_size": float(np.mean([v["train_size"] for v in outcomes])),
            "train_rmse": float(np.mean([v["train_rmse"] for v in outcomes])),
            "validation_rmse": float(np.mean([v["validation_rmse"] for v in outcomes])),
            "train_std": float(np.std([v["train_rmse"] for v in outcomes])),
            "validation_std": float(np.std([v["validation_rmse"] for v in outcomes])),
        })
    last = points[-1]
    relative_gap = (last["validation_rmse"] - last["train_rmse"]) / max(last["validation_rmse"], 1e-12)
    last_drop = points[-2]["validation_rmse"] - last["validation_rmse"] if len(points) > 1 else 0.0
    summary = {
        "train_rmse": last["train_rmse"], "val_rmse": last["validation_rmse"],
        "gap": last["validation_rmse"] - last["train_rmse"], "relative_gap": relative_gap,
        "last_step_validation_improvement": last_drop, "max_training_hours": common_hours,
        "max_training_examples_mean": last["train_size"],
        "trend_assessment": "A larger positive train-validation gap is consistent with variance, but temporal distribution shifts can also create a gap. A small gap alone does not establish high bias; compare the absolute error with the richer models. No diagnosis is forced by the model name.",
    }
    return {"points": points, "summary": summary}, raw_rows


def plot_curve_pair(curves, keys, titles, output_path):
    # B 的外推誤差可能很大，獨立 y 軸讓 A 的欠擬合曲線仍可讀；軸皆標示 RMSE。
    fig, axes = plt.subplots(1, 2, figsize=(11.4, 4.0), sharey=False)
    for ax, key, title in zip(axes, keys, titles):
        points = curves[key]["points"]
        n = np.array([p["train_size"] for p in points])
        for metric, std, label, color in [("train_rmse", "train_std", "Training", "#247f88"), ("validation_rmse", "validation_std", "Temporal validation", "#bc614a")]:
            mean = np.array([p[metric] for p in points])
            spread = np.array([p[std] for p in points])
            ax.plot(n, mean, marker="o", color=color, label=label)
            ax.fill_between(n, np.maximum(0, mean - spread), mean + spread, color=color, alpha=0.13)
        ax.set(title=title, xlabel="Training station-hour examples", ylabel="RMSE (original rental counts)")
        ax.grid(alpha=0.2)
        ax.legend()
    fig.tight_layout()
    _savefig(fig, output_path)


def _try_official_helper(helper, models, X, y, folds, output_dir):
    """老師 helper 僅供可選參考；呼叫結果不影響主流程是否完成。"""
    if helper is None:
        return {"state": "optional_not_used", "required": False, "message": "The optional course plot_learning_curve helper was not supplied. The independently implemented grouped temporal learning curves are the primary analysis. The TA announcement of 2026-10-19 permits continuing with one's own code; no starter/helper integration is required.", "calls": []}
    if not callable(helper):
        return {"state": "optional_skipped", "required": False, "message": "The optional official_learning_curve argument is not callable; the primary grouped temporal curves remain available.", "calls": []}
    try:
        signature = inspect.signature(helper)
    except (TypeError, ValueError) as exc:
        return {"state": "optional_skipped", "required": False, "message": f"Cannot inspect the optional helper signature: {exc}. The primary grouped temporal curves are unchanged.", "calls": []}
    params = signature.parameters
    # 若無法控制 CV、RMSE 與資料大小，不猜測老師函式的內部行為。
    if not all(name in params for name in ["cv", "train_sizes"]):
        return {"state": "optional_skipped", "required": False, "message": "The optional helper does not explicitly expose cv and train_sizes, so it was not called with an uncontrolled split. The primary analysis uses independently implemented grouped temporal curves.", "signature": str(signature), "calls": []}
    counts = X.groupby("datetime").size()
    if counts.nunique() != 1:
        return {"state": "optional_skipped", "required": False, "message": "Station counts vary by hour, so generic row-count train_sizes could split an hour. The optional helper was skipped; the primary implementation preserves complete-hour groups.", "calls": []}
    per_hour = int(counts.iloc[0])
    min_hours = min(len(h) for _, _, h in folds)
    sizes = np.unique(np.maximum(2, (np.array([0.1, 0.25, 0.5, 0.75, 1.0]) * min_hours).astype(int))) * per_hour
    cv = [(tr, va) for tr, va, _ in folds]
    calls = []
    for key, estimator in models.items():
        values = {"estimator": clone(estimator), "model": clone(estimator), "pipeline": clone(estimator), "X": X, "y": y, "title": key, "cv": cv, "train_sizes": sizes, "scoring": "neg_root_mean_squared_error", "n_jobs": 1, "random_state": SEED, "shuffle": False}
        kwargs = {name: values[name] for name in params if name in values}
        unsupported = [name for name, p in params.items() if p.default is inspect.Parameter.empty and p.kind not in (inspect.Parameter.VAR_POSITIONAL, inspect.Parameter.VAR_KEYWORD) and name not in kwargs]
        if unsupported:
            calls.append({"model": key, "status": "optional_skipped", "message": f"Optional helper arguments lack a mapping: {unsupported}"})
            continue
        try:
            plt.close("all")
            helper(**kwargs)
            if plt.get_fignums():
                relative = f"figures/official_{key}.png"
                plt.gcf().savefig(Path(output_dir) / relative, dpi=160, bbox_inches="tight")
            else:
                relative = None
            calls.append({"model": key, "status": "called_review_required", "figure": relative})
        except Exception as exc:
            calls.append({"model": key, "status": "optional_failed", "message": f"{type(exc).__name__}: {exc}"})
        finally:
            plt.close("all")
    return {"state": "optional_called" if calls and all(c["status"] == "called_review_required" for c in calls) else "optional_partial_or_failed", "required": False, "message": "The optional reference helper was called where its explicit signature allowed it; inspect any resulting reference plots before interpreting them. The primary grouped temporal curves use trailing complete-hour windows, while a generic helper may use prefixes. Neither use of the reference helper nor starter integration is required by the TA announcement of 2026-10-19.", "signature": str(signature), "calls": calls}


def _save_table(rows, name, output_dir, tables):
    relative = f"tables/{name}.csv"
    pd.DataFrame(rows).to_csv(Path(output_dir) / relative, index=False)
    tables[name] = relative


def run_assignment(data_dir=".", output_dir="a2_results", student_id="YOUR_ID", official_learning_curve=None, quick=False):
    """跑正式 CSV 的完整實驗；缺資料時在寫任何分數前停止。"""
    random.seed(SEED)
    np.random.seed(SEED)
    plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "axes.titlesize": 11, "figure.dpi": 110})
    train, fit, val, test, sample, data_summary = load_assignment_data(data_dir)
    out = Path(output_dir)
    (out / "tables").mkdir(parents=True, exist_ok=True)
    (out / "figures").mkdir(parents=True, exist_ok=True)
    X, y = fit[REQUIRED_X].copy(), fit["rentals"].copy()
    Xv, yv = val[REQUIRED_X].copy(), val["rentals"].copy()
    results = {
        "metadata": {
            "student_id": str(student_id), "seed": SEED, "status": "running",
            "generated_at": datetime.now(timezone.utc).isoformat(),
            "sklearn_version": sklearn.__version__, "numpy_version": np.__version__,
            "pandas_version": pd.__version__, "quick_mode": bool(quick),
            "official_starter_integration": "not_required_per_TA_2026_10_19",
            "assignment_notes": {
                "announcement_date": "2026-10-19",
                "source_url": "https://e3p.nycu.edu.tw/mod/forum/discuss.php?d=135396",
                "source_context": "TA announcement text provided by the user.",
                "summary": "The starter notebook is optional. Students who already have their own code may continue using it; the supplied data-loading, splitting and learning-curve helper code is for reference.",
                "starter_required": False,
                "reference_helper_required": False,
                "kaggle_team_name_format": "StudentID_Name",
                "existing_submission_naming_action": "If a submission already exists, rename the Kaggle team as required; a new submission is not required solely for this naming correction.",
            },
        },
        "data_summary": data_summary,
        "configuration": {
            "feature_descriptions": FEATURE_DESCRIPTIONS,
            "feature_comparison_sets": ["F0", "F1", "F2", "F3", "F4"],
            "feature_comparison_model": "Ridge", "feature_comparison_alpha": 10.0,
            "feature_comparison_scaler": "standard", "scaling_feature_set": "F2",
            "scaling_target": "raw", "numeric_scalers": ["none", "standard", "minmax"],
            "all_reported_metrics_original_scale": True,
            "part_ii_metrics_clipped": False, "selection_and_submission_clipped": True,
            "final_model_selection_policy": "First restrict final eligibility to stable model families using January-April forward-CV diagnostics. The unbounded degree-4 B family is diagnostic-only. Among eligible candidates, select the minimum May validation RMSE after nonnegative clipping; no June labels or leaderboard feedback are used.",
            "weather_bound_policy": "B_stable learns one minimum and maximum per weather column from the current fit partition, after training-only imputation and before scaling/polynomial expansion. Every CV fold and training size learns its own bounds. The final Jan-May refit relearns bounds from Jan-May only. No bounds are learned from June. No upper bound is placed on rental predictions and dock capacity is not used as a prediction cap.",
            "B_before_polynomial_degree": 4, "B_after_polynomial_degree": 2,
            "B_after_feature_set": "B_stable",
        },
        "files": {"figures": {}, "tables": {}, "submission": "submission.csv"},
        "leaderboard": {"test_rmse": None, "ta_baseline_rmse": None, "status": "pending_submission"},
        "limitations": ["Official test labels are unavailable; no test RMSE is calculated locally.", "Final model selection uses May validation only; leaderboard results must not be used to tune the model."] + data_summary["checks"],
    }
    tables = results["files"]["tables"]
    print("[1/7] 資料切分與 EDA：1–4 月訓練，5 月驗證，6 月測試。")
    eda, figs = make_eda(fit, out)
    results["files"]["figures"].update(figs)
    development_note = "An earlier official-data development run exposed a large training-versus-forward-validation gap for unbounded degree-4 weather terms within January-April. This motivated keeping that family for high-variance diagnosis only, adding F4 with linear weather, and testing a B remedy with degree-2 terms, fit-only weather-range bounds and Ridge. The eligible candidate set is explicit; final parameter/target selection within it uses May only. No June labels or leaderboard scores informed this change."
    results["diagnostics"] = {"eda": eda, "development_note": development_note}
    results["development_notes"] = {
        "motivation": development_note,
        "excluded_feature_families": ["B: unbounded degree-4 weather polynomial expansion"],
        "eligibility_rule": results["configuration"]["final_model_selection_policy"],
        "test_labels_used": False, "leaderboard_feedback_used": False,
        "training_curve_evidence": "Recorded below from this run's January-April diagnostic curves.",
    }

    print("[2/7] 原始天氣 baseline 與目標轉換比較。")
    target_rows = []
    for target in ["raw", "log1p"]:
        row, model = fit_and_evaluate(make_estimator("F0", "LinearRegression", scaler="none", target=target, quick=quick), X, y, Xv, yv)
        row.update(model="LinearRegression", target=target, feature_set="F0")
        target_rows.append(row)
    results["baseline"] = target_rows[0].copy()
    results["target_comparison"] = target_rows
    chosen_target = min(target_rows, key=lambda row: (row["val_rmse"], row["target"]))["target"]
    results["target_decision"] = {"selected": chosen_target, "criterion": "Lowest May RMSE in original rental counts, before non-negative clipping; fixed thereafter for cumulative feature comparison.", "note": "The raw weather LinearRegression baseline remains unchanged. The final model search separately compares raw and log targets for F3, F4 and B_stable."}
    _save_table([results["baseline"]], "baseline", out, tables)
    _save_table(target_rows, "target_comparison", out, tables)

    print("[3/7] F0–F4 累積特徵：固定 Ridge alpha=10、縮放與目標轉換。")
    feature_rows = []
    for feature_set in ["F0", "F1", "F2", "F3", "F4"]:
        row, _ = fit_and_evaluate(make_estimator(feature_set, "Ridge", alpha=10, target=chosen_target, quick=quick), X, y, Xv, yv)
        row.update(feature_set=feature_set, description=FEATURE_DESCRIPTIONS[feature_set], model="Ridge", alpha=10.0, scaler="standard", target=chosen_target)
        feature_rows.append(row)
    results["feature_comparison"] = feature_rows
    _save_table(feature_rows, "feature_comparison", out, tables)
    results["diagnostics"]["feature_gains"] = [{"from": a["feature_set"], "to": b["feature_set"], "validation_rmse_reduction": a["val_rmse"] - b["val_rmse"], "improved": b["val_rmse"] < a["val_rmse"]} for a, b in zip(feature_rows[:-1], feature_rows[1:])]

    print("[4/7] 3 種縮放 × 3 種模型；記錄 SGD epochs 與收斂警告。")
    scaling_rows = []
    for model_name in ["LinearRegression", "Ridge", "SGDRegressor"]:
        for scaler in ["none", "standard", "minmax"]:
            base = {"model": model_name, "scaler": scaler, "feature_set": "F2", "target": "raw", "alpha": 10.0 if model_name == "Ridge" else None}
            try:
                row, _ = fit_and_evaluate(make_estimator("F2", model_name, alpha=10, scaler=scaler, target="raw", quick=quick), X, y, Xv, yv)
                row.update(base)
                if model_name == "SGDRegressor":
                    row["max_epochs"] = 150 if quick else 2000
                    row["reached_max_epochs"] = row["epochs"] >= row["max_epochs"]
                scaling_rows.append(row)
            except Exception as exc:
                scaling_rows.append({**base, "val_rmse": None, "train_rmse": None, "val_r2": None, "epochs": None, "status": "failed", "warnings": [f"{type(exc).__name__}: {exc}"]})
    results["scaling_comparison"] = scaling_rows
    _save_table(scaling_rows, "scaling_comparison", out, tables)
    results["diagnostics"]["scaling"] = {
        "interpretation": "OLS should have approximately invariant predictions under equivalent invertible scaling with an intercept and stable numerical fitting. Fixed-alpha Ridge changes its effective penalty. SGD may change optimization behavior; its penalty is disabled to isolate that effect.",
        "sgd_settings": {"penalty": None, "learning_rate": "invscaling", "eta0": 0.01, "power_t": 0.25, "tol": 1e-3, "max_iter": 150 if quick else 2000, "random_state": SEED},
        "iteration_unit": "epochs (complete passes over training rows), not individual weight updates",
        "failed_combinations": sum(row["status"] == "failed" for row in scaling_rows),
    }

    print("[5/7] 使用 5 月選擇正則化強度與最終候選；尚未接觸任何 test 標籤。")
    tuning, fitted = [], {}
    candidates = [
        {"model_name": "weather_OLS_raw", "feature_set": "F0", "model": "LinearRegression", "alpha": None, "target": "raw", "stage": "final", "eligible_for_final_selection": True},
        {"model_name": "F3_OLS_raw", "feature_set": "F3", "model": "LinearRegression", "alpha": None, "target": "raw", "stage": "A_remedy", "eligible_for_final_selection": True},
        {"model_name": "B_OLS_raw", "feature_set": "B", "model": "LinearRegression", "alpha": None, "target": "raw", "stage": "diagnostic_only", "eligible_for_final_selection": False, "eligibility_reason": "Unbounded degree-4 weather expansion showed severe January-April forward-CV instability in development; retained for high-variance diagnosis, excluded from the stable final candidate set."},
    ]
    alpha_grid = [1.0, 100.0] if quick else [0.1, 1.0, 10.0, 100.0, 1000.0]
    for feature_set in ["F3", "F4", "B_stable"]:
        for target in ["raw", "log1p"]:
            for alpha in alpha_grid:
                candidates.append({"model_name": f"{feature_set}_Ridge_{target}_a{alpha:g}", "feature_set": feature_set, "model": "Ridge", "alpha": alpha, "target": target, "stage": "B_remedy" if feature_set == "B_stable" and target == "raw" else "final", "eligible_for_final_selection": True})
    for config in candidates:
        config.setdefault("eligibility_reason", "Eligible stable family: linear weather or bounded degree-2 weather terms; final ranking uses May clipped RMSE only.")
    for config in candidates:
        estimator = make_estimator(config["feature_set"], config["model"], alpha=config["alpha"] or 10, target=config["target"], quick=quick)
        try:
            row, estimator = fit_and_evaluate(estimator, X, y, Xv, yv)
            row.update(config)
            tuning.append(row)
            fitted[config["model_name"]] = estimator
        except Exception as exc:
            tuning.append({**config, "status": "failed", "val_rmse": None, "val_rmse_clipped": None, "val_r2_clipped": None, "warnings": [f"{type(exc).__name__}: {exc}"]})
    results["tuning"] = tuning
    _save_table(tuning, "tuning", out, tables)
    viable = [row for row in tuning if row["eligible_for_final_selection"] and row["status"] != "failed" and row["val_rmse_clipped"] is not None]
    if not viable:
        raise RuntimeError("All final candidates failed; inspect tuning.csv.")
    best = min(viable, key=lambda row: (row["val_rmse_clipped"], row["model_name"]))
    b_remedies = [row for row in viable if row["stage"] == "B_remedy"]
    if not b_remedies:
        raise RuntimeError("All bounded degree-2 B remedies failed; inspect tuning.csv.")
    best_b = min(b_remedies, key=lambda row: (row["val_rmse_clipped"], row["model_name"]))
    results["selection"] = {
        "model_name": best["model_name"], "model": best["model"], "feature_set": best["feature_set"], "alpha": best["alpha"], "target": best["target"], "scaler": "standard",
        "validation_rmse_raw": best["val_rmse"], "validation_rmse_clipped": best["val_rmse_clipped"], "validation_r2_clipped": best["val_r2_clipped"],
        "n_features": best["n_features"], "criterion": "Minimum May validation RMSE after nonnegative clipping among explicitly eligible stable candidates; no leaderboard scores used.",
        "eligibility_rule": results["configuration"]["final_model_selection_policy"],
        "B_remedy_alpha": best_b["alpha"], "B_remedy_feature_set": "B_stable",
        "B_remedy_target": "raw", "validation_negative_fraction": best["negative_prediction_fraction"],
        "weather_bounds_for_validation": learned_weather_bounds(fitted[best["model_name"]]),
        "weather_bounds_validation_fit_period": "January-April only",
    }
    # 實際選到的 May 預測圖；最後重訓後不能再把 May 當作未見資料重新打分。
    pred_val = np.maximum(fitted[best["model_name"]].predict(Xv), 0.0)
    plot_val = pd.DataFrame({"datetime": val["datetime"], "actual": yv, "prediction": pred_val})
    daily_val = plot_val.groupby(plot_val["datetime"].dt.date)[["actual", "prediction"]].mean()
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(pd.to_datetime(daily_val.index), daily_val["actual"], label="Actual")
    axes[0].plot(pd.to_datetime(daily_val.index), daily_val["prediction"], label="Prediction")
    axes[0].set(title="Selected model: May holdout", xlabel="Date", ylabel="Mean rentals per station-hour")
    axes[0].legend()
    axes[0].tick_params(axis="x", rotation=25)
    take = np.random.default_rng(SEED).choice(len(yv), size=min(5000, len(yv)), replace=False)
    axes[1].scatter(yv.iloc[take], pred_val[take], s=6, alpha=0.25)
    maximum = float(max(yv.max(), np.max(pred_val)))
    axes[1].plot([0, maximum], [0, maximum], "k--", linewidth=1)
    axes[1].set(title="May predictions (up to 5,000 points)", xlabel="Actual rentals", ylabel="Predicted rentals")
    fig.tight_layout()
    _savefig(fig, out / "figures/validation_predictions.png")
    results["files"]["figures"]["validation_predictions"] = "figures/validation_predictions.png"

    print("[6/7] A/B 改善前後學習曲線：Jan-Apr 完整小時、向前時間切分。")
    folds, fold_details = make_temporal_folds(X, n_splits=2 if quick else 3)
    model_specs = {
        "A_before": make_estimator("F0", "LinearRegression", target="raw", quick=quick),
        "B_before": make_estimator("B", "LinearRegression", target="raw", quick=quick),
        "A_after": make_estimator("F3", "LinearRegression", target="raw", quick=quick),
        "B_after": make_estimator("B_stable", "Ridge", alpha=best_b["alpha"], target="raw", quick=quick),
    }
    curves, curve_rows = {}, []
    for key, estimator in model_specs.items():
        print(f"  {key}")
        curves[key], raw = compute_learning_curve(estimator, X, y, folds, quick=quick)
        curves[key]["figure"] = "figures/learning_before.png" if key.endswith("before") else "figures/learning_after.png"
        curves[key]["description"] = {"A_before": "Weather-only OLS (deliberately simple).", "B_before": "Fine categorical interactions and unbounded degree-4 weather terms, OLS without regularization; diagnostic-only.", "A_after": "Add time, station and selected interactions to OLS; raw target unchanged.", "B_after": f"Retain fine categorical interactions, reduce weather expansion from degree 4 to degree 2, clip weather inputs to bounds learned within each fit partition, and use Ridge alpha={best_b['alpha']:g}. Raw target is unchanged; rental predictions have no upper cap."}[key]
        curve_rows += [{"model": key, **row} for row in raw]
    plot_curve_pair(curves, ["A_before", "B_before"], ["A: weather-only OLS", "B: flexible OLS"], out / "figures/learning_before.png")
    plot_curve_pair(curves, ["A_after", "B_after"], ["A remedy: richer features", "B remedy: bounded degree-2 + Ridge"], out / "figures/learning_after.png")
    results["files"]["figures"].update(learning_before="figures/learning_before.png", learning_after="figures/learning_after.png")
    _save_table(curve_rows, "learning_curves", out, tables)
    _write_json(out / "temporal_cv.json", fold_details)
    helper_status = _try_official_helper(official_learning_curve, model_specs, X, y, folds, out)
    for call in helper_status.get("calls", []):
        if call.get("figure") and (out / call["figure"]).is_file():
            results["files"]["figures"]["official_" + call["model"]] = call["figure"]
    results["learning_curves"] = {"helper_status": helper_status, "cv_description": "Forward chronological validation within January-April, split on unique datetime groups. Every size uses complete hours and the most recent training hours ending at the same fold boundary. All scalers/encoders are refitted in each fold and size. Curves use raw, unclipped rental-count RMSE.", "folds": fold_details, "models": curves, "size_note": "The largest common training-hour count across folds is used so the x-axis has comparable sizes. The final point need not include all January-April rows."}
    results["learning_curves"]["implementation"] = "self_implemented_grouped_temporal_learning_curves"
    results["learning_curves"]["official_helper_required"] = False
    results["learning_curves"]["remedy_description"] = curves["B_after"]["description"]
    results["learning_curves"]["figure_axis_note"] = "Each side-by-side panel has its own labeled RMSE axis so the unstable B curve does not flatten the visible A curve. Compare numerical RMSE values, not vertical pixel distances across panels."
    results["learning_curves"]["regularization_selection_note"] = "B remedy alpha is chosen on May validation before these Jan-Apr diagnostic curves are evaluated. The curves are descriptive diagnostics for that selected setting, not nested-CV estimates of model-selection performance."
    interpretations = {}
    for prefix in ["A", "B"]:
        before, after = curves[prefix + "_before"]["summary"], curves[prefix + "_after"]["summary"]
        gain = before["val_rmse"] - after["val_rmse"]
        text = f"At the largest common training size, validation RMSE changed from {before['val_rmse']:.4f} to {after['val_rmse']:.4f} ({'improved' if gain > 0 else 'did not improve'}). Training RMSE changed from {before['train_rmse']:.4f} to {after['train_rmse']:.4f}; the validation-minus-training gap changed from {before['gap']:.4f} to {after['gap']:.4f}."
        if prefix == "A":
            text += " A small gap plus a clear improvement from richer features supports a high-bias explanation. If richer features do not help, high bias is not established by these results. More data alone may have limited value once both curves plateau."
        else:
            text += " A large original gap that shrinks under the combined remedy supports a high-variance/extrapolation explanation. The remedy combines lower polynomial degree, training-only weather-range bounds and Ridge, so the gain cannot be attributed to regularization alone. If this pattern is absent, do not claim that B overfits simply because it is flexible. Additional representative data may help when the validation curve is still falling; temporal shift remains another explanation."
        interpretations[prefix] = text
    results["diagnostics"]["learning_curve_interpretations"] = interpretations
    results["development_notes"]["training_curve_evidence"] = {
        "source": "This run's grouped forward-CV diagnostics within January-April; no June targets.",
        "B_before_unbounded_degree4": curves["B_before"]["summary"],
        "B_after_bounded_degree2_Ridge": curves["B_after"]["summary"],
    }

    print("[7/7] 最終以 1–5 月重訓，按 sample_submission 的 id 順序輸出。")
    final_model = clone(fitted[best["model_name"]])
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always")
        final_model.fit(train[REQUIRED_X], train["rentals"])
        predictions = np.asarray(final_model.predict(test[REQUIRED_X]), dtype=float).reshape(-1)
    submission = construct_submission(test, sample, predictions)
    target_column = data_summary["prediction_column"]
    submission.to_csv(out / "submission.csv", index=False)
    results["submission_summary"] = {"rows": len(submission), "id_column": data_summary["submission_id_column"], "prediction_column": target_column, "min_prediction": float(submission[target_column].min()), "max_prediction": float(submission[target_column].max()), "negative_predictions": 0, "id_order_matches_sample": True, "training_rows": len(train), "warnings": list(dict.fromkeys(str(w.message) for w in captured)), "note": "Predictions are nonnegative floating-point trip counts; they are neither rounded nor capped by dock capacity."}
    results["submission_summary"]["weather_bounds_final_refit"] = learned_weather_bounds(final_model)
    results["submission_summary"]["weather_bounds_fit_period"] = "January-May only; June features do not fit or update the bounds."
    results["metadata"]["status"] = "experiments_complete_pending_leaderboard"
    if quick:
        results["limitations"].append("Quick mode used fewer candidate alphas, CV folds and training sizes, and a smaller SGD epoch budget. Re-run the intended full configuration before final submission.")
    _write_json(out / "results.json", results)
    print(f"已產生輸入 CSV 的結果與 submission.csv：{out.resolve()}")
    print("自行撰寫的程式可直接續用；starter 與老師 helper 為可選參考。")
    print("Kaggle 隊名須為 StudentID_Name；如已提交，僅為修正隊名不必重傳預測。")
    print("尚需取得排行榜 test RMSE，並填入報告的學號與 Colab 連結。")
    return _json_safe(results)


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Run A2 experiments on the official course CSV files.")
    parser.add_argument("--data-dir", default=".")
    parser.add_argument("--output-dir", default="a2_results")
    parser.add_argument("--student-id", default="YOUR_ID")
    parser.add_argument("--quick", action="store_true", help="Reduced computation for debugging; full run is recommended for submission.")
    args = parser.parse_args()
    run_assignment(args.data_dir, args.output_dir, args.student_id, quick=args.quick)


Writing a2_pipeline.py


In [ ]:
%%writefile a2_report.py
"""A2 PDF reports: a completed analytical solution and a results-driven report.

Only Part I can be completed without the official course data.  The full report
reads measured results from ``a2_pipeline.run_assignment``; it never invents
scores, diagnoses, leaderboard results, or a Colab URL.

Usage:
    python a2_report.py --part1 --output A2_PartI_Solutions.pdf
    python a2_report.py --results a2_results --student-id YOUR_ID \
        --output A2_YOUR_ID.pdf
"""

from __future__ import annotations

import argparse
import base64
import io
import json
import math
import os
import re
import tempfile
from pathlib import Path
from xml.sax.saxutils import escape

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "a2_matplotlib"))
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import inch
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.platypus import (
    BaseDocTemplate, Frame, Image, KeepTogether, PageBreak, PageTemplate,
    Paragraph, Spacer, Table, TableStyle,
)


NAVY = colors.HexColor("#17324D")
TEAL = colors.HexColor("#087F8C")
MUTED = colors.HexColor("#536476")
LIGHT = colors.HexColor("#EEF4F8")
GOLD = colors.HexColor("#AE6725")
PAGE_W, PAGE_H = A4
MARGIN = 43
CONTENT_W = PAGE_W - 2 * MARGIN
TA_NOTE_DATE = "2026-10-19"
TA_NOTE_URL = "https://e3p.nycu.edu.tw/mod/forum/discuss.php?d=135396"
KAGGLE_INVITE_URL = "https://www.kaggle.com/t/93a788c887e34269939b30da23305dfe"
# Transcribed from the actual baseline.txt supplied in A2_data_and_baseline_1_6.
# These are source quotations, never substituted for a recomputed metric.
OFFICIAL_BASELINE_REFERENCE = {
    "source": "baseline.txt in the supplied A2_data_and_baseline_1_6 dataset",
    "train_rmse": 25.0947, "train_r2": 0.1437,
    "val_rmse": 29.3875, "val_r2": 0.1437,
    "test_rmse": 24.2461, "test_r2": 0.1263,
}


# Noto Sans TC, weight 400, subset to the verified student's name glyphs.
# Source: https://github.com/google/fonts/tree/main/ofl/notosanstc
# Copyright 2014-2021 Adobe; SIL Open Font License 1.1. The full license and
# copyright notices are retained in the embedded font's name table (ID 13).
# Renamed A2 Name CJK Subset; SHA-256 of the subset:
# 28a5c804a72b0996729cfe506257eb3088a8e7d13aa606167cff1fc8ec7b77ad
# Keeping this small subset in the module makes the notebook portable offline.
_CJK_NAME_FONT_B64 = (
    "AAEAAAASAQAABAAgQkFTRYwGwEgAAC50AAABCkdQT1NEdkx1AAAvgAAAACBHU1VCbTRZLgAAL6AAAABCT1MvMlU83swAAAGoAAAA"
    "YFNUQVR4kmzdAAAv5AAAAC5jbWFwUAnavAAAAhgAAABMZ2FzcAAAABAAAC5sAAAACGdseWb0Ub6UAAACeAAAAvhoZWFkJq/VLgAA"
    "ASwAAAA2aGhlYQhyAusAAAFkAAAAJGhtdHgIzwC+AAACCAAAABBsb2NhAVECHQAAAmwAAAAMbWF4cAAYAG8AAAGIAAAAIG5hbWVZ"
    "k+PeAAAFcAAAKNxwb3N0/4YAMgAALkwAAAAgcHJlcGgGjIUAAAJkAAAAB3ZoZWEF3xH1AAAwFAAAACR2bXR4B6oAUAAAMDgAAAAM"
    "AAEAAAACAQYM664/Xw889QADA+gAAAAA3LAl+QAAAADm/2pUAB//iAPJA3AAAAAGAAIAAAAAAAAAAQAABIj+4AAAA+gAHwAfA8kA"
    "AQAAAAAAAAAAAAAAAAAAAAMAAQAAAAUAbgASAAAAAAABAAAAAAAAAAAAAAAAAAAAAAAEA9kBkAAFAAACigJYAAAASwKKAlgAAAFe"
    "ADIBRQAAAgsCAAAAAAAAAAAAAAEIAAAAAAAAAAAAAABBREJPAEAAIIPvA3D/iAAABIgBIAAAAAEAAAAAAh8C3QAAACAABgPoAGQA"
    "4AAAA+gAJwAfADMAAAACAAAAAwAAABQAAwABAAAAFAAEADgAAAAKAAgAAgACACBPc1a0g+///wAAACBPc1a0g+/////hsI+pT3wV"
    "AAEAAAAAAAAAAAAAuAH/hbAEjQAAAAArACsAdgEmAXwABQBk/4gDhANwAAMABgAJAAwADwAAVxEhEQEBIQEBEQEhAQERAWQDIP5w"
    "AT79hAFeAT79ZAJ8/sL+ogE+eAPo/BgCHQGZ/j7+ZwMy/KUBmQHC/M4BmQAIACf/sAPAA0gADAARABUAGQAdACEAJQApAABBFw4C"
    "By4CJz4CBzcXESMTIRUhAyEVIQMhFSEDIRUhATMRIxUzESMBDEYeUF8zAxARBy5VSEtJAUrRAgf9+RQCLP3UHAJk/ZwiApr9ZgEs"
    "TExMTANEFVSlkjkJHB0IMYCTvUkB/SwDFUf+j0cBKUj+d0gDcv59Rf5rAAASAB//rQPJAx4AAwAHAAsADwATABcAGwAfACcAMgA9"
    "AEkATQBdAGEAZQBpAG0AAFMhFSEFIRUhBTMVIxUzFSMDIRUhNzMVIyczESMTMxEjJTY2NxUGBgcBFwYGBy4CJzY2FxcGBgcuAic2"
    "NiceAhcGBgcuAicBIRUhJzMVFA4CBy4CJz4CNRMVMzUnIRUhJRUzNSchFSHuAUf+uQGmAR7+4v6N3d3d3VABev6GnEFBbD4+3D8/"
    "/tZAsWBYrEMB4EIURS4FExMIKz27RR+YgQMREgh8jIgYUnFHDBsJSHRUGv4oAxf86SRFBxQmHwYVFggmJAp1re8BNv7KAc+v8gE5"
    "/scB6jMUPVktPywBCTWNawX+ngFg/kJvBQ4HNQgRBgIfDFmgNwYREQUxjy0IrOJEBxQUBjrQblSSbh8KIw0kfKJdAQhAQOEzdHt0"
    "MQgREAM9kJE9AX1HRy2idUdHLaIADAAz/68DtgNIAAMABwALAA8AEwAXABsAHwAjACcAKwAvAABTIRUhByEVIRMhFSETIRUhATMR"
    "IwMzFSMlMxUjATMVIyUzFSMlIRUhJSEVIRMhFSFNAU/+sRoDg/x9GgNQ/LAYAxn85wFmS0vuRkYB6UdH/lpJSQFZSUn9wwGk/lwB"
    "ygGl/ltEAU/+sQGzQmVCAYtA/mBAAgf9cwJL1tbWAiTX19ePQ0ND/vlCAAAAEgDeAAMAAQQJAAAAnAAAAAMAAQQJAAEAJACcAAMA"
    "AQQJAAIADgDAAAMAAQQJAAMANgDOAAMAAQQJAAQANAEEAAMAAQQJAAUAagE4AAMAAQQJAAYALgGiAAMAAQQJAAcAlgHQAAMAAQQJ"
    "AAgACgJmAAMAAQQJAAkBlAJwAAMAAQQJAAoBCgQEAAMAAQQJAAsANAUOAAMAAQQJAA0iSAVCAAMAAQQJAA4ANCeKAAMAAQQJABAA"
    "GCe+AAMAAQQJABEACCfWAAMAAQQJABkAFCfeAAMAAQQJAQEADCfyACgAYwApACAAMgAwADEANAAtADIAMAAyADEAIABBAGQAbwBi"
    "AGUAIAAoAGgAdAB0AHAAOgAvAC8AdwB3AHcALgBhAGQAbwBiAGUALgBjAG8AbQAvACkALAAgAHcAaQB0AGgAIABSAGUAcwBlAHIA"
    "dgBlAGQAIABGAG8AbgB0ACAATgBhAG0AZQAgACcAUwBvAHUAcgBjAGUAJwAuAEEAMgAgAE4AYQBtAGUAIABDAEoASwAgAFMAdQBi"
    "AHMAZQB0AFIAZQBnAHUAbABhAHIAQQAyAE4AYQBtAGUAQwBKAEsAUwB1AGIAcwBlAHQALQBSAGUAZwB1AGwAYQByAC0AMQAuADAA"
    "QQAyACAATgBhAG0AZQAgAEMASgBLACAAUwB1AGIAcwBlAHQAIABSAGUAZwB1AGwAYQByAFYAZQByAHMAaQBvAG4AIAAyAC4AMAAw"
    "ADQALQBIADIAOwBoAG8AdABjAG8AbgB2ACAAMQAuADAALgAxADEAOAA7AG0AYQBrAGUAbwB0AGYAZQB4AGUAIAAyAC4ANQAuADYA"
    "NQA2ADAAMwBBADIATgBhAG0AZQBDAEoASwBTAHUAYgBzAGUAdAAtAFIAZQBnAHUAbABhAHIAUwBvAHUAcgBjAGUAIABpAHMAIABh"
    "ACAAdAByAGEAZABlAG0AYQByAGsAIABvAGYAIABBAGQAbwBiAGUAIABpAG4AIAB0AGgAZQAgAFUAbgBpAHQAZQBkACAAUwB0AGEA"
    "dABlAHMAIABhAG4AZAAvAG8AcgAgAG8AdABoAGUAcgAgAGMAbwB1AG4AdAByAGkAZQBzAC4AQQBkAG8AYgBlAFIAeQBvAGsAbwAg"
    "AE4ASQBTAEgASQBaAFUASwBBACCJf1habbxbUAAgACgAawBhAG4AYQAsACAAYgBvAHAAbwBtAG8AZgBvACAAJgAgAGkAZABlAG8A"
    "ZwByAGEAcABoAHMAKQA7ACAAUABhAHUAbAAgAEQALgAgAEgAdQBuAHQAIAAoAEwAYQB0AGkAbgAsACAARwByAGUAZQBrACAAJgAg"
    "AEMAeQByAGkAbABsAGkAYwApADsAIABTAGEAbgBkAG8AbABsACAAQwBvAG0AbQB1AG4AaQBjAGEAdABpAG8AbgBzACDAsLPMzuS7"
    "pLLIzwDHdMFYACwAIABTAG8AbwAtAHkAbwB1AG4AZwAgAEoAQQBOAEcAIMelwhjGAQAgACYAIABKAG8AbwAtAHkAZQBvAG4AIABL"
    "AEEATgBHACCsFcj8xfAAIAAoAGgAYQBuAGcAdQBsACAAZQBsAGUAbQBlAG4AdABzACwAIABsAGUAdAB0AGUAcgBzACAAJgAgAHMA"
    "eQBsAGwAYQBiAGwAZQBzACkARAByAC4AIABLAGUAbgAgAEwAdQBuAGQAZQAgACgAcAByAG8AagBlAGMAdAAgAGEAcgBjAGgAaQB0"
    "AGUAYwB0ACwAIABnAGwAeQBwAGgAIABzAGUAdAAgAGQAZQBmAGkAbgBpAHQAaQBvAG4AIAAmACAAbwB2AGUAcgBhAGwAbAAgAHAA"
    "cgBvAGQAdQBjAHQAaQBvAG4AKQA7ACAATQBhAHMAYQB0AGEAawBhACAASABBAFQAVABPAFIASQAgZw2Q6GtjjLQAIAAoAHAAcgBv"
    "AGQAdQBjAHQAaQBvAG4AIAAmACAAaQBkAGUAbwBnAHIAYQBwAGgAIABlAGwAZQBtAGUAbgB0AHMAKQBoAHQAdABwADoALwAvAHcA"
    "dwB3AC4AYQBkAG8AYgBlAC4AYwBvAG0ALwB0AHkAcABlAC8AQwBvAHAAeQByAGkAZwBoAHQAIAAyADAAMQA0AC0AMgAwADIAMQAg"
    "AEEAZABvAGIAZQAgACgAaAB0AHQAcAA6AC8ALwB3AHcAdwAuAGEAZABvAGIAZQAuAGMAbwBtAC8AKQAsACAAdwBpAHQAaAAgAFIA"
    "ZQBzAGUAcgB2AGUAZAAgAEYAbwBuAHQAIABOAGEAbQBlACAAJwBTAG8AdQByAGMAZQAnAAoACgBUAGgAaQBzACAARgBvAG4AdAAg"
    "AFMAbwBmAHQAdwBhAHIAZQAgAGkAcwAgAGwAaQBjAGUAbgBzAGUAZAAgAHUAbgBkAGUAcgAgAHQAaABlACAAUwBJAEwAIABPAHAA"
    "ZQBuACAARgBvAG4AdAAgAEwAaQBjAGUAbgBzAGUALAAgAFYAZQByAHMAaQBvAG4AIAAxAC4AMQAuAAoAVABoAGkAcwAgAGwAaQBj"
    "AGUAbgBzAGUAIABpAHMAIABjAG8AcABpAGUAZAAgAGIAZQBsAG8AdwAsACAAYQBuAGQAIABpAHMAIABhAGwAcwBvACAAYQB2AGEA"
    "aQBsAGEAYgBsAGUAIAB3AGkAdABoACAAYQAgAEYAQQBRACAAYQB0ADoACgBoAHQAdABwAHMAOgAvAC8AcwBjAHIAaQBwAHQAcwAu"
    "AHMAaQBsAC4AbwByAGcALwBPAEYATAAKAAoACgAtAC0ALQAtAC0ALQAtAC0ALQAtAC0ALQAtAC0ALQAtAC0ALQAtAC0ALQAtAC0A"
    "LQAtAC0ALQAtAC0ALQAtAC0ALQAtAC0ALQAtAC0ALQAtAC0ALQAtAC0ALQAtAC0ALQAtAC0ALQAtAC0ALQAtAC0ALQAtAC0ACgBT"
    "AEkATAAgAE8AUABFAE4AIABGAE8ATgBUACAATABJAEMARQBOAFMARQAgAFYAZQByAHMAaQBvAG4AIAAxAC4AMQAgAC0AIAAyADYA"
    "IABGAGUAYgByAHUAYQByAHkAIAAyADAAMAA3AAoALQAtAC0ALQAtAC0ALQAtAC0ALQAtAC0ALQAtAC0ALQAtAC0ALQAtAC0ALQAt"
    "AC0ALQAtAC0ALQAtAC0ALQAtAC0ALQAtAC0ALQAtAC0ALQAtAC0ALQAtAC0ALQAtAC0ALQAtAC0ALQAtAC0ALQAtAC0ALQAtAAoA"
    "CgBQAFIARQBBAE0AQgBMAEUACgBUAGgAZQAgAGcAbwBhAGwAcwAgAG8AZgAgAHQAaABlACAATwBwAGUAbgAgAEYAbwBuAHQAIABM"
    "AGkAYwBlAG4AcwBlACAAKABPAEYATAApACAAYQByAGUAIAB0AG8AIABzAHQAaQBtAHUAbABhAHQAZQAgAHcAbwByAGwAZAB3AGkA"
    "ZABlAAoAZABlAHYAZQBsAG8AcABtAGUAbgB0ACAAbwBmACAAYwBvAGwAbABhAGIAbwByAGEAdABpAHYAZQAgAGYAbwBuAHQAIABw"
    "AHIAbwBqAGUAYwB0AHMALAAgAHQAbwAgAHMAdQBwAHAAbwByAHQAIAB0AGgAZQAgAGYAbwBuAHQAIABjAHIAZQBhAHQAaQBvAG4A"
    "CgBlAGYAZgBvAHIAdABzACAAbwBmACAAYQBjAGEAZABlAG0AaQBjACAAYQBuAGQAIABsAGkAbgBnAHUAaQBzAHQAaQBjACAAYwBv"
    "AG0AbQB1AG4AaQB0AGkAZQBzACwAIABhAG4AZAAgAHQAbwAgAHAAcgBvAHYAaQBkAGUAIABhACAAZgByAGUAZQAgAGEAbgBkAAoA"
    "bwBwAGUAbgAgAGYAcgBhAG0AZQB3AG8AcgBrACAAaQBuACAAdwBoAGkAYwBoACAAZgBvAG4AdABzACAAbQBhAHkAIABiAGUAIABz"
    "AGgAYQByAGUAZAAgAGEAbgBkACAAaQBtAHAAcgBvAHYAZQBkACAAaQBuACAAcABhAHIAdABuAGUAcgBzAGgAaQBwAAoAdwBpAHQA"
    "aAAgAG8AdABoAGUAcgBzAC4ACgAKAFQAaABlACAATwBGAEwAIABhAGwAbABvAHcAcwAgAHQAaABlACAAbABpAGMAZQBuAHMAZQBk"
    "ACAAZgBvAG4AdABzACAAdABvACAAYgBlACAAdQBzAGUAZAAsACAAcwB0AHUAZABpAGUAZAAsACAAbQBvAGQAaQBmAGkAZQBkACAA"
    "YQBuAGQACgByAGUAZABpAHMAdAByAGkAYgB1AHQAZQBkACAAZgByAGUAZQBsAHkAIABhAHMAIABsAG8AbgBnACAAYQBzACAAdABo"
    "AGUAeQAgAGEAcgBlACAAbgBvAHQAIABzAG8AbABkACAAYgB5ACAAdABoAGUAbQBzAGUAbAB2AGUAcwAuACAAVABoAGUACgBmAG8A"
    "bgB0AHMALAAgAGkAbgBjAGwAdQBkAGkAbgBnACAAYQBuAHkAIABkAGUAcgBpAHYAYQB0AGkAdgBlACAAdwBvAHIAawBzACwAIABj"
    "AGEAbgAgAGIAZQAgAGIAdQBuAGQAbABlAGQALAAgAGUAbQBiAGUAZABkAGUAZAAsACAACgByAGUAZABpAHMAdAByAGkAYgB1AHQA"
    "ZQBkACAAYQBuAGQALwBvAHIAIABzAG8AbABkACAAdwBpAHQAaAAgAGEAbgB5ACAAcwBvAGYAdAB3AGEAcgBlACAAcAByAG8AdgBp"
    "AGQAZQBkACAAdABoAGEAdAAgAGEAbgB5ACAAcgBlAHMAZQByAHYAZQBkAAoAbgBhAG0AZQBzACAAYQByAGUAIABuAG8AdAAgAHUA"
    "cwBlAGQAIABiAHkAIABkAGUAcgBpAHYAYQB0AGkAdgBlACAAdwBvAHIAawBzAC4AIABUAGgAZQAgAGYAbwBuAHQAcwAgAGEAbgBk"
    "ACAAZABlAHIAaQB2AGEAdABpAHYAZQBzACwACgBoAG8AdwBlAHYAZQByACwAIABjAGEAbgBuAG8AdAAgAGIAZQAgAHIAZQBsAGUA"
    "YQBzAGUAZAAgAHUAbgBkAGUAcgAgAGEAbgB5ACAAbwB0AGgAZQByACAAdAB5AHAAZQAgAG8AZgAgAGwAaQBjAGUAbgBzAGUALgAg"
    "AFQAaABlAAoAcgBlAHEAdQBpAHIAZQBtAGUAbgB0ACAAZgBvAHIAIABmAG8AbgB0AHMAIAB0AG8AIAByAGUAbQBhAGkAbgAgAHUA"
    "bgBkAGUAcgAgAHQAaABpAHMAIABsAGkAYwBlAG4AcwBlACAAZABvAGUAcwAgAG4AbwB0ACAAYQBwAHAAbAB5AAoAdABvACAAYQBu"
    "AHkAIABkAG8AYwB1AG0AZQBuAHQAIABjAHIAZQBhAHQAZQBkACAAdQBzAGkAbgBnACAAdABoAGUAIABmAG8AbgB0AHMAIABvAHIA"
    "IAB0AGgAZQBpAHIAIABkAGUAcgBpAHYAYQB0AGkAdgBlAHMALgAKAAoARABFAEYASQBOAEkAVABJAE8ATgBTAAoAIgBGAG8AbgB0"
    "ACAAUwBvAGYAdAB3AGEAcgBlACIAIAByAGUAZgBlAHIAcwAgAHQAbwAgAHQAaABlACAAcwBlAHQAIABvAGYAIABmAGkAbABlAHMA"
    "IAByAGUAbABlAGEAcwBlAGQAIABiAHkAIAB0AGgAZQAgAEMAbwBwAHkAcgBpAGcAaAB0AAoASABvAGwAZABlAHIAKABzACkAIAB1"
    "AG4AZABlAHIAIAB0AGgAaQBzACAAbABpAGMAZQBuAHMAZQAgAGEAbgBkACAAYwBsAGUAYQByAGwAeQAgAG0AYQByAGsAZQBkACAA"
    "YQBzACAAcwB1AGMAaAAuACAAVABoAGkAcwAgAG0AYQB5AAoAaQBuAGMAbAB1AGQAZQAgAHMAbwB1AHIAYwBlACAAZgBpAGwAZQBz"
    "ACwAIABiAHUAaQBsAGQAIABzAGMAcgBpAHAAdABzACAAYQBuAGQAIABkAG8AYwB1AG0AZQBuAHQAYQB0AGkAbwBuAC4ACgAKACIA"
    "UgBlAHMAZQByAHYAZQBkACAARgBvAG4AdAAgAE4AYQBtAGUAIgAgAHIAZQBmAGUAcgBzACAAdABvACAAYQBuAHkAIABuAGEAbQBl"
    "AHMAIABzAHAAZQBjAGkAZgBpAGUAZAAgAGEAcwAgAHMAdQBjAGgAIABhAGYAdABlAHIAIAB0AGgAZQAKAGMAbwBwAHkAcgBpAGcA"
    "aAB0ACAAcwB0AGEAdABlAG0AZQBuAHQAKABzACkALgAKAAoAIgBPAHIAaQBnAGkAbgBhAGwAIABWAGUAcgBzAGkAbwBuACIAIABy"
    "AGUAZgBlAHIAcwAgAHQAbwAgAHQAaABlACAAYwBvAGwAbABlAGMAdABpAG8AbgAgAG8AZgAgAEYAbwBuAHQAIABTAG8AZgB0AHcA"
    "YQByAGUAIABjAG8AbQBwAG8AbgBlAG4AdABzACAAYQBzAAoAZABpAHMAdAByAGkAYgB1AHQAZQBkACAAYgB5ACAAdABoAGUAIABD"
    "AG8AcAB5AHIAaQBnAGgAdAAgAEgAbwBsAGQAZQByACgAcwApAC4ACgAKACIATQBvAGQAaQBmAGkAZQBkACAAVgBlAHIAcwBpAG8A"
    "bgAiACAAcgBlAGYAZQByAHMAIAB0AG8AIABhAG4AeQAgAGQAZQByAGkAdgBhAHQAaQB2AGUAIABtAGEAZABlACAAYgB5ACAAYQBk"
    "AGQAaQBuAGcAIAB0AG8ALAAgAGQAZQBsAGUAdABpAG4AZwAsAAoAbwByACAAcwB1AGIAcwB0AGkAdAB1AHQAaQBuAGcAIAAtAC0A"
    "IABpAG4AIABwAGEAcgB0ACAAbwByACAAaQBuACAAdwBoAG8AbABlACAALQAtACAAYQBuAHkAIABvAGYAIAB0AGgAZQAgAGMAbwBt"
    "AHAAbwBuAGUAbgB0AHMAIABvAGYAIAB0AGgAZQAKAE8AcgBpAGcAaQBuAGEAbAAgAFYAZQByAHMAaQBvAG4ALAAgAGIAeQAgAGMA"
    "aABhAG4AZwBpAG4AZwAgAGYAbwByAG0AYQB0AHMAIABvAHIAIABiAHkAIABwAG8AcgB0AGkAbgBnACAAdABoAGUAIABGAG8AbgB0"
    "ACAAUwBvAGYAdAB3AGEAcgBlACAAdABvACAAYQAKAG4AZQB3ACAAZQBuAHYAaQByAG8AbgBtAGUAbgB0AC4ACgAKACIAQQB1AHQA"
    "aABvAHIAIgAgAHIAZQBmAGUAcgBzACAAdABvACAAYQBuAHkAIABkAGUAcwBpAGcAbgBlAHIALAAgAGUAbgBnAGkAbgBlAGUAcgAs"
    "ACAAcAByAG8AZwByAGEAbQBtAGUAcgAsACAAdABlAGMAaABuAGkAYwBhAGwACgB3AHIAaQB0AGUAcgAgAG8AcgAgAG8AdABoAGUA"
    "cgAgAHAAZQByAHMAbwBuACAAdwBoAG8AIABjAG8AbgB0AHIAaQBiAHUAdABlAGQAIAB0AG8AIAB0AGgAZQAgAEYAbwBuAHQAIABT"
    "AG8AZgB0AHcAYQByAGUALgAKAAoAUABFAFIATQBJAFMAUwBJAE8ATgAgACYAIABDAE8ATgBEAEkAVABJAE8ATgBTAAoAUABlAHIA"
    "bQBpAHMAcwBpAG8AbgAgAGkAcwAgAGgAZQByAGUAYgB5ACAAZwByAGEAbgB0AGUAZAAsACAAZgByAGUAZQAgAG8AZgAgAGMAaABh"
    "AHIAZwBlACwAIAB0AG8AIABhAG4AeQAgAHAAZQByAHMAbwBuACAAbwBiAHQAYQBpAG4AaQBuAGcACgBhACAAYwBvAHAAeQAgAG8A"
    "ZgAgAHQAaABlACAARgBvAG4AdAAgAFMAbwBmAHQAdwBhAHIAZQAsACAAdABvACAAdQBzAGUALAAgAHMAdAB1AGQAeQAsACAAYwBv"
    "AHAAeQAsACAAbQBlAHIAZwBlACwAIABlAG0AYgBlAGQALAAgAG0AbwBkAGkAZgB5ACwACgByAGUAZABpAHMAdAByAGkAYgB1AHQA"
    "ZQAsACAAYQBuAGQAIABzAGUAbABsACAAbQBvAGQAaQBmAGkAZQBkACAAYQBuAGQAIAB1AG4AbQBvAGQAaQBmAGkAZQBkACAAYwBv"
    "AHAAaQBlAHMAIABvAGYAIAB0AGgAZQAgAEYAbwBuAHQACgBTAG8AZgB0AHcAYQByAGUALAAgAHMAdQBiAGoAZQBjAHQAIAB0AG8A"
    "IAB0AGgAZQAgAGYAbwBsAGwAbwB3AGkAbgBnACAAYwBvAG4AZABpAHQAaQBvAG4AcwA6AAoACgAxACkAIABOAGUAaQB0AGgAZQBy"
    "ACAAdABoAGUAIABGAG8AbgB0ACAAUwBvAGYAdAB3AGEAcgBlACAAbgBvAHIAIABhAG4AeQAgAG8AZgAgAGkAdABzACAAaQBuAGQA"
    "aQB2AGkAZAB1AGEAbAAgAGMAbwBtAHAAbwBuAGUAbgB0AHMALAAKAGkAbgAgAE8AcgBpAGcAaQBuAGEAbAAgAG8AcgAgAE0AbwBk"
    "AGkAZgBpAGUAZAAgAFYAZQByAHMAaQBvAG4AcwAsACAAbQBhAHkAIABiAGUAIABzAG8AbABkACAAYgB5ACAAaQB0AHMAZQBsAGYA"
    "LgAKAAoAMgApACAATwByAGkAZwBpAG4AYQBsACAAbwByACAATQBvAGQAaQBmAGkAZQBkACAAVgBlAHIAcwBpAG8AbgBzACAAbwBm"
    "ACAAdABoAGUAIABGAG8AbgB0ACAAUwBvAGYAdAB3AGEAcgBlACAAbQBhAHkAIABiAGUAIABiAHUAbgBkAGwAZQBkACwACgByAGUA"
    "ZABpAHMAdAByAGkAYgB1AHQAZQBkACAAYQBuAGQALwBvAHIAIABzAG8AbABkACAAdwBpAHQAaAAgAGEAbgB5ACAAcwBvAGYAdAB3"
    "AGEAcgBlACwAIABwAHIAbwB2AGkAZABlAGQAIAB0AGgAYQB0ACAAZQBhAGMAaAAgAGMAbwBwAHkACgBjAG8AbgB0AGEAaQBuAHMA"
    "IAB0AGgAZQAgAGEAYgBvAHYAZQAgAGMAbwBwAHkAcgBpAGcAaAB0ACAAbgBvAHQAaQBjAGUAIABhAG4AZAAgAHQAaABpAHMAIABs"
    "AGkAYwBlAG4AcwBlAC4AIABUAGgAZQBzAGUAIABjAGEAbgAgAGIAZQAKAGkAbgBjAGwAdQBkAGUAZAAgAGUAaQB0AGgAZQByACAA"
    "YQBzACAAcwB0AGEAbgBkAC0AYQBsAG8AbgBlACAAdABlAHgAdAAgAGYAaQBsAGUAcwAsACAAaAB1AG0AYQBuAC0AcgBlAGEAZABh"
    "AGIAbABlACAAaABlAGEAZABlAHIAcwAgAG8AcgAKAGkAbgAgAHQAaABlACAAYQBwAHAAcgBvAHAAcgBpAGEAdABlACAAbQBhAGMA"
    "aABpAG4AZQAtAHIAZQBhAGQAYQBiAGwAZQAgAG0AZQB0AGEAZABhAHQAYQAgAGYAaQBlAGwAZABzACAAdwBpAHQAaABpAG4AIAB0"
    "AGUAeAB0ACAAbwByAAoAYgBpAG4AYQByAHkAIABmAGkAbABlAHMAIABhAHMAIABsAG8AbgBnACAAYQBzACAAdABoAG8AcwBlACAA"
    "ZgBpAGUAbABkAHMAIABjAGEAbgAgAGIAZQAgAGUAYQBzAGkAbAB5ACAAdgBpAGUAdwBlAGQAIABiAHkAIAB0AGgAZQAgAHUAcwBl"
    "AHIALgAKAAoAMwApACAATgBvACAATQBvAGQAaQBmAGkAZQBkACAAVgBlAHIAcwBpAG8AbgAgAG8AZgAgAHQAaABlACAARgBvAG4A"
    "dAAgAFMAbwBmAHQAdwBhAHIAZQAgAG0AYQB5ACAAdQBzAGUAIAB0AGgAZQAgAFIAZQBzAGUAcgB2AGUAZAAgAEYAbwBuAHQACgBO"
    "AGEAbQBlACgAcwApACAAdQBuAGwAZQBzAHMAIABlAHgAcABsAGkAYwBpAHQAIAB3AHIAaQB0AHQAZQBuACAAcABlAHIAbQBpAHMA"
    "cwBpAG8AbgAgAGkAcwAgAGcAcgBhAG4AdABlAGQAIABiAHkAIAB0AGgAZQAgAGMAbwByAHIAZQBzAHAAbwBuAGQAaQBuAGcACgBD"
    "AG8AcAB5AHIAaQBnAGgAdAAgAEgAbwBsAGQAZQByAC4AIABUAGgAaQBzACAAcgBlAHMAdAByAGkAYwB0AGkAbwBuACAAbwBuAGwA"
    "eQAgAGEAcABwAGwAaQBlAHMAIAB0AG8AIAB0AGgAZQAgAHAAcgBpAG0AYQByAHkAIABmAG8AbgB0ACAAbgBhAG0AZQAgAGEAcwAK"
    "AHAAcgBlAHMAZQBuAHQAZQBkACAAdABvACAAdABoAGUAIAB1AHMAZQByAHMALgAKAAoANAApACAAVABoAGUAIABuAGEAbQBlACgA"
    "cwApACAAbwBmACAAdABoAGUAIABDAG8AcAB5AHIAaQBnAGgAdAAgAEgAbwBsAGQAZQByACgAcwApACAAbwByACAAdABoAGUAIABB"
    "AHUAdABoAG8AcgAoAHMAKQAgAG8AZgAgAHQAaABlACAARgBvAG4AdAAKAFMAbwBmAHQAdwBhAHIAZQAgAHMAaABhAGwAbAAgAG4A"
    "bwB0ACAAYgBlACAAdQBzAGUAZAAgAHQAbwAgAHAAcgBvAG0AbwB0AGUALAAgAGUAbgBkAG8AcgBzAGUAIABvAHIAIABhAGQAdgBl"
    "AHIAdABpAHMAZQAgAGEAbgB5AAoATQBvAGQAaQBmAGkAZQBkACAAVgBlAHIAcwBpAG8AbgAsACAAZQB4AGMAZQBwAHQAIAB0AG8A"
    "IABhAGMAawBuAG8AdwBsAGUAZABnAGUAIAB0AGgAZQAgAGMAbwBuAHQAcgBpAGIAdQB0AGkAbwBuACgAcwApACAAbwBmACAAdABo"
    "AGUACgBDAG8AcAB5AHIAaQBnAGgAdAAgAEgAbwBsAGQAZQByACgAcwApACAAYQBuAGQAIAB0AGgAZQAgAEEAdQB0AGgAbwByACgA"
    "cwApACAAbwByACAAdwBpAHQAaAAgAHQAaABlAGkAcgAgAGUAeABwAGwAaQBjAGkAdAAgAHcAcgBpAHQAdABlAG4ACgBwAGUAcgBt"
    "AGkAcwBzAGkAbwBuAC4ACgAKADUAKQAgAFQAaABlACAARgBvAG4AdAAgAFMAbwBmAHQAdwBhAHIAZQAsACAAbQBvAGQAaQBmAGkA"
    "ZQBkACAAbwByACAAdQBuAG0AbwBkAGkAZgBpAGUAZAAsACAAaQBuACAAcABhAHIAdAAgAG8AcgAgAGkAbgAgAHcAaABvAGwAZQAs"
    "AAoAbQB1AHMAdAAgAGIAZQAgAGQAaQBzAHQAcgBpAGIAdQB0AGUAZAAgAGUAbgB0AGkAcgBlAGwAeQAgAHUAbgBkAGUAcgAgAHQA"
    "aABpAHMAIABsAGkAYwBlAG4AcwBlACwAIABhAG4AZAAgAG0AdQBzAHQAIABuAG8AdAAgAGIAZQAKAGQAaQBzAHQAcgBpAGIAdQB0"
    "AGUAZAAgAHUAbgBkAGUAcgAgAGEAbgB5ACAAbwB0AGgAZQByACAAbABpAGMAZQBuAHMAZQAuACAAVABoAGUAIAByAGUAcQB1AGkA"
    "cgBlAG0AZQBuAHQAIABmAG8AcgAgAGYAbwBuAHQAcwAgAHQAbwAKAHIAZQBtAGEAaQBuACAAdQBuAGQAZQByACAAdABoAGkAcwAg"
    "AGwAaQBjAGUAbgBzAGUAIABkAG8AZQBzACAAbgBvAHQAIABhAHAAcABsAHkAIAB0AG8AIABhAG4AeQAgAGQAbwBjAHUAbQBlAG4A"
    "dAAgAGMAcgBlAGEAdABlAGQACgB1AHMAaQBuAGcAIAB0AGgAZQAgAEYAbwBuAHQAIABTAG8AZgB0AHcAYQByAGUALgAKAAoAVABF"
    "AFIATQBJAE4AQQBUAEkATwBOAAoAVABoAGkAcwAgAGwAaQBjAGUAbgBzAGUAIABiAGUAYwBvAG0AZQBzACAAbgB1AGwAbAAgAGEA"
    "bgBkACAAdgBvAGkAZAAgAGkAZgAgAGEAbgB5ACAAbwBmACAAdABoAGUAIABhAGIAbwB2AGUAIABjAG8AbgBkAGkAdABpAG8AbgBz"
    "ACAAYQByAGUACgBuAG8AdAAgAG0AZQB0AC4ACgAKAEQASQBTAEMATABBAEkATQBFAFIACgBUAEgARQAgAEYATwBOAFQAIABTAE8A"
    "RgBUAFcAQQBSAEUAIABJAFMAIABQAFIATwBWAEkARABFAEQAIAAiAEEAUwAgAEkAUwAiACwAIABXAEkAVABIAE8AVQBUACAAVwBB"
    "AFIAUgBBAE4AVABZACAATwBGACAAQQBOAFkAIABLAEkATgBEACwACgBFAFgAUABSAEUAUwBTACAATwBSACAASQBNAFAATABJAEUA"
    "RAAsACAASQBOAEMATABVAEQASQBOAEcAIABCAFUAVAAgAE4ATwBUACAATABJAE0ASQBUAEUARAAgAFQATwAgAEEATgBZACAAVwBB"
    "AFIAUgBBAE4AVABJAEUAUwAgAE8ARgAKAE0ARQBSAEMASABBAE4AVABBAEIASQBMAEkAVABZACwAIABGAEkAVABOAEUAUwBTACAA"
    "RgBPAFIAIABBACAAUABBAFIAVABJAEMAVQBMAEEAUgAgAFAAVQBSAFAATwBTAEUAIABBAE4ARAAgAE4ATwBOAEkATgBGAFIASQBO"
    "AEcARQBNAEUATgBUAAoATwBGACAAQwBPAFAAWQBSAEkARwBIAFQALAAgAFAAQQBUAEUATgBUACwAIABUAFIAQQBEAEUATQBBAFIA"
    "SwAsACAATwBSACAATwBUAEgARQBSACAAUgBJAEcASABUAC4AIABJAE4AIABOAE8AIABFAFYARQBOAFQAIABTAEgAQQBMAEwAIABU"
    "AEgARQAKAEMATwBQAFkAUgBJAEcASABUACAASABPAEwARABFAFIAIABCAEUAIABMAEkAQQBCAEwARQAgAEYATwBSACAAQQBOAFkA"
    "IABDAEwAQQBJAE0ALAAgAEQAQQBNAEEARwBFAFMAIABPAFIAIABPAFQASABFAFIAIABMAEkAQQBCAEkATABJAFQAWQAsAAoASQBO"
    "AEMATABVAEQASQBOAEcAIABBAE4AWQAgAEcARQBOAEUAUgBBAEwALAAgAFMAUABFAEMASQBBAEwALAAgAEkATgBEAEkAUgBFAEMA"
    "VAAsACAASQBOAEMASQBEAEUATgBUAEEATAAsACAATwBSACAAQwBPAE4AUwBFAFEAVQBFAE4AVABJAEEATAAKAEQAQQBNAEEARwBF"
    "AFMALAAgAFcASABFAFQASABFAFIAIABJAE4AIABBAE4AIABBAEMAVABJAE8ATgAgAE8ARgAgAEMATwBOAFQAUgBBAEMAVAAsACAA"
    "VABPAFIAVAAgAE8AUgAgAE8AVABIAEUAUgBXAEkAUwBFACwAIABBAFIASQBTAEkATgBHAAoARgBSAE8ATQAsACAATwBVAFQAIABP"
    "AEYAIABUAEgARQAgAFUAUwBFACAATwBSACAASQBOAEEAQgBJAEwASQBUAFkAIABUAE8AIABVAFMARQAgAFQASABFACAARgBPAE4A"
    "VAAgAFMATwBGAFQAVwBBAFIARQAgAE8AUgAgAEYAUgBPAE0ACgBPAFQASABFAFIAIABEAEUAQQBMAEkATgBHAFMAIABJAE4AIABU"
    "AEgARQAgAEYATwBOAFQAIABTAE8ARgBUAFcAQQBSAEUALgAKAGgAdAB0AHAAOgAvAC8AcwBjAHIAaQBwAHQAcwAuAHMAaQBsAC4A"
    "bwByAGcALwBPAEYATABOAG8AdABvACAAUwBhAG4AcwAgAFQAQwBUAGgAaQBuAE4AbwB0AG8AUwBhAG4AcwBUAEMAVwBlAGkAZwBo"
    "AHQAAwAAAAAAAP+DADIAAAAAAAAAAAAAAAAAAAAAAAAAAAABAAH//wAPAAEAAQAMAGoAAADqAGIABAAGREZMVAAmY3lybAA4Z3Jl"
    "awA4aGFuaQAma2FuYQAmbGF0bgA4AAYAAAAAAAIABAAeACQAKgCmAAYAAAAAAAMABAAMABIAGACUAAP/vQB2AAMDOwB8AAH/iAAE"
    "ABYABGljZmJpY2Z0aWRlb3JvbW4ABkRGTFQAJmN5cmwAOGdyZWsAOGhhbmkAJmthbmEAJmxhdG4AOAAGAAAAAAACAAQAHgAqADYA"
    "OgAGAAAAAAADAAQADAAYACQAKAADADUABgAAAACAAAADA7MABgAAAAGAAAABAAAAAQB4AAEAAAAMAAEAAAAWAAEAAQAAQABAAAAC"
    "AAAAAQAA5RsAAAABAAAACgAcAB4AAURGTFQACAAEAAAAAP//AAAAAAAAAAEAAAAKAD4AQAAGREZMVAAmY3lybAAwZ3JlawAwaGFu"
    "aQAwa2FuYQAwbGF0bgAwAAQAAAAA//8AAAAAAAAAAAAAAAAAAQABAAgAAQAAABQAAQAAABwAAndnaHQBAQAAAAIAAwAAAAIAAgGQ"
    "AAACvAAAAAAAARAAAfT+DAAAA+gAAAAAA+gAAAABAAAAAAAAAAAAAAAAAAED6AAAA3AAKABSACg="
)


def _register_fonts():
    """Embed fonts; never depend on a PDF viewer supplying a CJK font."""
    if "A2NameCJK" not in pdfmetrics.getRegisteredFontNames():
        pdfmetrics.registerFont(TTFont(
            "A2NameCJK", io.BytesIO(base64.b64decode(_CJK_NAME_FONT_B64))
        ))
    candidates = [
        Path(matplotlib.get_data_path()) / "fonts" / "ttf",
        Path("/usr/share/fonts/truetype/dejavu"),
    ]
    for root in candidates:
        if all((root / filename).is_file() for filename in
               ["DejaVuSans.ttf", "DejaVuSans-Bold.ttf", "DejaVuSans-Oblique.ttf"]):
            for name, filename in [
                ("A2Sans", "DejaVuSans.ttf"),
                ("A2SansBold", "DejaVuSans-Bold.ttf"),
                ("A2SansOblique", "DejaVuSans-Oblique.ttf"),
            ]:
                if name not in pdfmetrics.getRegisteredFontNames():
                    pdfmetrics.registerFont(TTFont(name, str(root / filename)))
            pdfmetrics.registerFontFamily(
                "A2Sans", normal="A2Sans", bold="A2SansBold", italic="A2SansOblique"
            )
            return
    raise RuntimeError("DejaVu Sans is required; install matplotlib or DejaVu fonts.")


def _styles():
    _register_fonts()
    styles = getSampleStyleSheet()
    styles.add(ParagraphStyle(
        name="A2Body", fontName="A2Sans", fontSize=9.4, leading=14.4,
        textColor=NAVY, spaceAfter=7,
    ))
    styles.add(ParagraphStyle(
        name="A2Small", parent=styles["A2Body"], fontSize=8, leading=11.8,
        textColor=MUTED, spaceAfter=5,
    ))
    styles.add(ParagraphStyle(
        name="A2Title", fontName="A2SansBold", fontSize=22, leading=27,
        textColor=NAVY, spaceAfter=9,
    ))
    styles.add(ParagraphStyle(
        name="A2H1", fontName="A2SansBold", fontSize=16, leading=21,
        textColor=NAVY, spaceBefore=4, spaceAfter=11, keepWithNext=True,
    ))
    styles.add(ParagraphStyle(
        name="A2H2", fontName="A2SansBold", fontSize=11.2, leading=16,
        textColor=TEAL, spaceBefore=9, spaceAfter=6, keepWithNext=True,
    ))
    styles.add(ParagraphStyle(
        name="A2Label", fontName="A2SansBold", fontSize=8.2, leading=11,
        textColor=TEAL, spaceAfter=5,
    ))
    styles.add(ParagraphStyle(
        name="A2Cell", fontName="A2Sans", fontSize=8, leading=11.5,
        textColor=NAVY,
    ))
    styles.add(ParagraphStyle(
        name="A2Header", parent=styles["A2Cell"], fontName="A2SansBold",
        textColor=colors.white,
    ))
    styles.add(ParagraphStyle(
        name="A2Caption", parent=styles["A2Small"], alignment=TA_CENTER,
        spaceAfter=9,
    ))
    return styles


def _p(text, styles, kind="A2Body", raw=False):
    markup = text if raw else escape(str(text))
    # The supplied student's three Chinese name glyphs use an embedded font,
    # while the report's existing Latin typography and layout remain unchanged.
    markup = re.sub(r"[嚴佳華]+", lambda match: '<font name="A2NameCJK">'
                    + match.group(0) + '</font>', markup)
    return Paragraph(markup, styles[kind])


def _table(headers, rows, styles, widths=None):
    def cell(value, header=False):
        if value is None:
            value = "Pending / N/A"
        return _p(value, styles, "A2Header" if header else "A2Cell")
    data = [[cell(h, True) for h in headers]]
    data.extend([[cell(c) for c in row] for row in rows])
    t = Table(data, colWidths=widths, repeatRows=1, hAlign="LEFT")
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), NAVY),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, LIGHT]),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING", (0, 0), (-1, -1), 8),
        ("RIGHTPADDING", (0, 0), (-1, -1), 8),
        ("TOPPADDING", (0, 0), (-1, -1), 5),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
        ("LINEBELOW", (0, 0), (-1, 0), 0.6, TEAL),
        ("LINEBELOW", (0, -1), (-1, -1), 0.4, colors.HexColor("#CAD5DF")),
    ]))
    return t


def _callout(text, styles):
    t = Table([[_p(text, styles, raw=True)]], colWidths=[CONTENT_W])
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), LIGHT),
        ("BOX", (0, 0), (-1, -1), 0.5, colors.HexColor("#C7D8E2")),
        ("LEFTPADDING", (0, 0), (-1, -1), 11),
        ("RIGHTPADDING", (0, 0), (-1, -1), 11),
        ("TOPPADDING", (0, 0), (-1, -1), 9),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 3),
    ]))
    return t


def _equation(tex, directory, idx, max_width=CONTENT_W, size=15):
    # Matplotlib mathtext supports bold groups but not LaTeX's boxed command.
    tex = tex.replace(r"\boxed", r"\mathbf")
    fig = plt.figure(figsize=(7.0, 0.50))
    fig.patch.set_alpha(0)
    fig.text(.5, .5, "$" + tex + "$", ha="center", va="center", fontsize=size,
             color="#17324D")
    path = Path(directory) / f"equation_{idx}.png"
    fig.savefig(path, dpi=220, transparent=True, bbox_inches="tight", pad_inches=.07)
    plt.close(fig)
    im = Image(str(path))
    scale = min(max_width / im.imageWidth, 0.36)
    im.drawWidth = im.imageWidth * scale
    im.drawHeight = im.imageHeight * scale
    im.hAlign = "CENTER"
    return im


def _chart_image(path, max_height=315):
    im = Image(str(path))
    scale = min(CONTENT_W / im.imageWidth, max_height / im.imageHeight)
    im.drawWidth = im.imageWidth * scale
    im.drawHeight = im.imageHeight * scale
    im.hAlign = "CENTER"
    return im


def _toy_chart(directory):
    x = np.array([0, 1, 2, 3, 4], dtype=float)
    y = np.array([10, 8, 7, 4, 1], dtype=float)
    xplot = np.linspace(0, 4, 200)
    fig, ax = plt.subplots(figsize=(7.2, 3.1))
    ax.scatter(x, y, color="#17324D", s=38, zorder=4, label="Observed rentals")
    ax.plot(xplot, 10.4 - 2.2 * xplot, color="#087F8C", lw=2.2,
            label="OLS (either unit)")
    ax.plot(xplot, 10 - 2 * xplot, color="#4567B0", ls="--", lw=1.8,
            label="Ridge, mm, lambda=1")
    ax.plot(xplot, 6.4 - .2 * xplot, color="#BF742E", ls="-.", lw=1.8,
            label="Ridge, cm, lambda=1 (shown per mm)")
    ax.set(xlabel="Precipitation (mm)", ylabel="Rentals (trips)",
           xlim=(-.1, 4.1), ylim=(0, 11.1))
    ax.grid(alpha=.18)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=8, loc="lower left")
    fig.tight_layout()
    path = Path(directory) / "toy_fits.png"
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return path


def _header_footer(canvas, doc):
    canvas.saveState()
    canvas.setStrokeColor(TEAL)
    canvas.setLineWidth(1.0)
    canvas.line(MARGIN, PAGE_H - 29, PAGE_W - MARGIN, PAGE_H - 29)
    canvas.setFont("A2Sans", 7.5)
    canvas.setFillColor(MUTED)
    canvas.drawString(MARGIN, PAGE_H - 21, "MACHINE LEARNING  |  ASSIGNMENT 2: THE MORNING RUSH")
    canvas.line(MARGIN, 31, PAGE_W - MARGIN, 31)
    canvas.drawString(MARGIN, 20, "Source: A2 handout. Analytical calculations and measured results are distinguished.")
    canvas.drawRightString(PAGE_W - MARGIN, 20, str(doc.page))
    canvas.restoreState()


def _document(output_pdf, title, author="Student solution"):
    output_pdf = Path(output_pdf)
    output_pdf.parent.mkdir(parents=True, exist_ok=True)
    doc = BaseDocTemplate(
        str(output_pdf), pagesize=A4, leftMargin=MARGIN, rightMargin=MARGIN,
        topMargin=46, bottomMargin=43, title=title,
        author=author,
    )
    frame = Frame(MARGIN, 43, CONTENT_W, PAGE_H - 89, id="main", topPadding=0,
                  bottomPadding=0, leftPadding=0, rightPadding=0)
    doc.addPageTemplates(PageTemplate(id="report", frames=[frame], onPage=_header_footer))
    return doc


def _part1_story(styles, directory, student_id="YOUR_ID", standalone=True):
    s = []
    eq_idx = 0
    def eq(tex, size=15):
        nonlocal eq_idx
        eq_idx += 1
        s.extend([_equation(tex, directory, eq_idx, size=size), Spacer(1, 6)])
    def body(text):
        s.append(_p(text, styles, raw=True))
    def h2(text):
        s.append(_p(text, styles, "A2H2"))

    s.append(_p("PART I  /  ANALYTICAL SOLUTION", styles, "A2Label"))
    s.append(_p("Least squares by hand", styles, "A2Title"))
    body("<b>Student ID:</b> " + escape(student_id) + " &nbsp;&nbsp; <b>Random seed:</b> 0 where applicable.")
    s.append(_p("The handout (p. 1) permits handwritten or typeset solutions. "
                "This typeset solution includes the required derivations and calculations.",
                styles, "A2Small"))
    if standalone:
        s.append(_callout(
            "<b>Completed component:</b> all Part I calculations and explanations. "
            "The empirical Parts II and III require the official CSV files and measured experiments; "
            "this PDF does not claim that those experiments have been run. The starter notebook "
            "and plotting helper are optional references under the TA clarification of 19 October 2026.", styles))
        s.append(Spacer(1, 9))
    h2("1. Ridge normal equations")
    body("For the stated matrix objective, every coefficient in w is penalized:")
    eq(r"J(w)=\frac{1}{2}\|y-Xw\|_2^2+\frac{\lambda}{2}\|w\|_2^2,\quad \lambda>0")
    body("Differentiate with respect to w and set the gradient to zero:")
    eq(r"\nabla_w J=X^\top(Xw-y)+\lambda w=0")
    eq(r"(X^\top X+\lambda I)w=X^\top y")
    eq(r"\boxed{w^*=(X^\top X+\lambda I)^{-1}X^\top y}")
    h2("2. Why the solution is unique")
    body("For any nonzero vector v, the quadratic form is strictly positive:")
    eq(r"v^\top(X^\top X+\lambda I)v=\|Xv\|_2^2+\lambda\|v\|_2^2>0", size=14)
    body("Thus the Hessian is positive definite, the objective is strictly convex, and "
         "the minimizer is unique. This remains true when X<super>T</super>X is singular, "
         "because the positive penalty supplies curvature in every coefficient direction.")
    h2("3. Intercept convention for the toy data")
    body("The handout specifies an <b>unpenalized intercept</b> for the toy regression. "
         "If a constant column is included in X, use the penalty matrix "
         "P = diag(0, 1, ..., 1), rather than I. Equivalently, center x and y, fit the "
         "penalized slope, then recover w<sub>0</sub> = y-bar − w<sub>1</sub>x-bar. "
         "For a single unpenalized intercept and at least one observation, with all "
         "slopes penalized, the solution is still unique.")
    h2("Centered quantities used in the numerical solution")
    body("For the five observations listed on the next page, x-bar = 10/5 = 2 and "
         "y-bar = 30/5 = 6. The centered x values are (−2, −1, 0, 1, 2); the centered "
         "y values are (4, 2, 1, −2, −5). Thus S<sub>xx</sub> = 4+1+0+1+4 = 10, "
         "S<sub>xy</sub> = −8−2+0−2−10 = −22, and S<sub>yy</sub> = 16+4+1+4+25 = 50.")

    s.append(PageBreak())
    s.append(_p("PART I(a)  /  NUMERICAL CALCULATIONS", styles, "A2Label"))
    s.append(_p("OLS and Ridge in millimetres", styles, "A2H1"))
    s.append(_table(["Hour", "1", "2", "3", "4", "5"], [
        ["x (mm)", 0, 1, 2, 3, 4], ["y (trips)", 10, 8, 7, 4, 1]], styles,
        [104] + [(CONTENT_W-104)/5]*5))
    s.append(Spacer(1, 8))
    body("The sample means and centered sums are:")
    eq(r"\bar{x}=2,\quad \bar{y}=6,\quad S_{xx}=10,\quad S_{xy}=-22,\quad S_{yy}=50", size=14)
    h2("4. Ordinary least squares")
    eq(r"w_1=\frac{S_{xy}}{S_{xx}}=\frac{-22}{10}=-2.2,\quad w_0=6-(-2.2)(2)=10.4", size=13)
    eq(r"\boxed{\hat{y}=10.4-2.2x}")
    s.append(_table(["x (mm)", "Observed y", "Predicted y", "Residual y − prediction", "Squared residual"], [
        [0, 10, "10.4", "−0.4", "0.16"],
        [1, 8, "8.2", "−0.2", "0.04"],
        [2, 7, "6.0", "1.0", "1.00"],
        [3, 4, "3.8", "0.2", "0.04"],
        [4, 1, "1.6", "−0.6", "0.36"],
    ], styles, [64, 82, 86, 156, CONTENT_W-388]))
    s.append(Spacer(1, 7))
    eq(r"\mathrm{SSE}=1.6,\qquad \boxed{\mathrm{MSE}=1.6/5=0.32}")
    eq(r"\boxed{R^2=1-\frac{1.6}{50}=0.968}")
    body("MSE divides by N = 5. The denominator N − 2 belongs to a residual-variance "
         "estimate, not to the requested mean squared prediction error.")
    h2("5. Ridge shrinkage")
    eq(r"w_1(\lambda)=\frac{-22}{10+\lambda},\qquad w_0=6-2w_1")
    s.append(_table(["Method", "Intercept", "Slope (trips/mm)"], [
        ["OLS", "10.4", "−2.2"],
        ["Ridge, lambda = 1", "10.0", "−2.0"],
        ["Ridge, lambda = 10", "8.2", "−1.1"],
    ], styles, [CONTENT_W*.46, CONTENT_W*.22, CONTENT_W*.32]))
    s.append(Spacer(1, 7))
    body("As lambda tends to infinity, the slope tends to zero and the unpenalized "
         "intercept tends to 6. All predictions therefore approach the mean rental count.")

    s.append(PageBreak())
    s.append(_p("PART I(b)  /  CHANGE OF UNITS", styles, "A2Label"))
    s.append(_p("Changing millimetres to centimetres", styles, "A2H1"))
    body("Define z = x / 10. The new feature values are 0, 0.1, 0.2, 0.3, 0.4 cm.")
    eq(r"\bar{z}=0.2,\qquad S_{zz}=S_{xx}/100=0.1,\qquad S_{zy}=S_{xy}/10=-2.2", size=13)
    h2("6. OLS predictions do not change")
    eq(r"w_{1,\mathrm{cm}}=\frac{-2.2}{0.1}=-22,\qquad w_0=6-(-22)(0.2)=10.4", size=13)
    eq(r"10.4-22z=10.4-22(x/10)=10.4-2.2x")
    body("The numerical slope is ten times larger because one centimetre is ten "
         "millimetres. The predicted rental counts are identical, so MSE remains "
         "0.32 and R² remains 0.968. The fitted relationship has not changed.")
    h2("7. Ridge with the same lambda is different")
    eq(r"w_{1,\mathrm{cm}}=\frac{-2.2}{0.1+1}=-2,\qquad w_0=6-(-2)(0.2)=6.4", size=13)
    body("The new slope is −2 trips per centimetre. Expressed as an effect per "
         "millimetre, it is −2 / 10 = <b>−0.2 trips/mm</b>, rather than the original "
         "Ridge slope of −2 trips/mm.")
    s.append(_table(["Fit", "Intercept", "Slope per stated unit", "Effect per mm"], [
        ["OLS on mm", "10.4", "−2.2 / mm", "−2.2"],
        ["OLS on cm", "10.4", "−22 / cm", "−2.2"],
        ["Ridge on mm, lambda=1", "10.0", "−2 / mm", "−2.0"],
        ["Ridge on cm, lambda=1", "6.4", "−2 / cm", "−0.2"],
    ], styles, [190, 70, 139, CONTENT_W-399]))
    s.append(Spacer(1, 9))
    body("The new Ridge predictions, in hour order, are (6.4, 6.2, 6.0, 5.8, 5.6). "
         "They are more strongly shrunk toward the mean than the original "
         "millimetre-based predictions (10, 8, 6, 4, 2).")
    h2("8. Why the penalty changes")
    body("To keep the same physical effect after dividing x by 10, the numerical "
         "coefficient must be multiplied by 10. Its square is then multiplied by 100. "
         "Keeping lambda fixed therefore imposes a stronger penalty on the same "
         "physical relationship. Ridge is not invariant to feature scaling at fixed lambda.")
    eq(r"\boxed{\lambda_{\mathrm{cm}}=\lambda_{\mathrm{mm}}/100}")
    body("For example, lambda = 0.01 in centimetres recovers the original "
         "lambda = 1 millimetre solution: a slope of −20 trips/cm (= −2 trips/mm) "
         "and an intercept of 10. This is a property of the single-feature rescaling "
         "used here; independently rescaling several features generally requires "
         "feature-specific penalty adjustments to preserve the same objective.")

    s.append(PageBreak())
    s.append(_p("PART I(b)  /  OPTIMIZATION AND INTERPRETATION", styles, "A2Label"))
    s.append(_p("Scaling and gradient descent", styles, "A2H1"))
    h2("9. A common learning rate must handle every direction")
    body("For a quadratic loss with Hessian H, batch gradient descent obeys "
         "w<sub>t+1</sub> = w<sub>t</sub> − eta ∇J(w<sub>t</sub>). "
         "Writing the parameter error as e<sub>t</sub> = w<sub>t</sub> − w*, "
         "the error update is e<sub>t+1</sub> = (I − eta H)e<sub>t</sub>.")
    eq(r"H\succ0:\qquad 0<\eta<\frac{2}{\lambda_{\max}(H)}")
    body("Large differences in feature scale can create very different curvatures "
         "across parameter directions. A learning rate large enough to move quickly "
         "along a flat direction can oscillate or diverge along a steep direction. "
         "A small rate that stabilizes the steep direction can make progress in the "
         "flat direction very slow.")
    body("Scaling often improves conditioning and makes a shared learning rate more "
         "effective. It does not guarantee convergence for every learning-rate choice, "
         "and unscaled data do not inevitably diverge. Strong feature correlations "
         "may also leave the problem poorly conditioned after standardization.")
    h2("10. Visual check of the analytical fits")
    s.append(_chart_image(_toy_chart(directory), max_height=247))
    s.append(_p(
        "All curves above are computed from the five observations in the handout. "
        "Both axes use original physical units, so the difference caused by keeping "
        "Ridge lambda fixed is directly comparable.", styles, "A2Caption"))
    s.append(_callout(
        "<b>Conclusions:</b> OLS predictions are unchanged by this unit conversion. "
        "At fixed lambda, Ridge shrinks the physical rainfall effect differently. "
        "Feature scaling also changes the geometry encountered by gradient descent.", styles))
    s.append(Spacer(1, 11))
    body("<b>Source and scope.</b> Toy observations, objective, and task requirements: "
         "A2_Handout.pdf, Part I, page 2. All numerical values here are analytical "
         "calculations, not empirical estimates from the separate YouBike CSV dataset.")
    return s


def build_part1(output_pdf, student_id="YOUR_ID"):
    """Create only the fully solved analytical component (four pages)."""
    styles = _styles()
    with tempfile.TemporaryDirectory(prefix="a2_report_") as directory:
        story = _part1_story(styles, directory, student_id=student_id)
        _document(output_pdf, "A2 - Part I analytical solutions").build(story)
    return str(Path(output_pdf).resolve())


def _num(value, digits=4):
    if value is None:
        return "N/A"
    try:
        value = float(value)
    except (TypeError, ValueError):
        return str(value)
    if not math.isfinite(value):
        return "Not finite"
    if abs(value) >= 10_000_000:
        return f"{value:.{digits}e}"
    return f"{value:,.{digits}f}"


def _interpretation_paragraphs(value, styles, label=None):
    """Render the pipeline's measured diagnostic statements without fabricating any."""
    story = []
    if label:
        story.append(_p(label, styles, "A2H2"))
    if isinstance(value, dict):
        for k, v in value.items():
            title = str(k).replace("_", " ").capitalize()
            if isinstance(v, (list, dict)):
                story.extend(_interpretation_paragraphs(v, styles, title))
            else:
                story.append(_p(f"{title}: {v}", styles))
    elif isinstance(value, list):
        for i, v in enumerate(value):
            if isinstance(v, (list, dict)):
                story.extend(_interpretation_paragraphs(v, styles))
            else:
                story.append(_p(str(v), styles))
    elif value:
        story.append(_p(str(value), styles))
    return story


def build_report(results_dir, output_pdf, student_id="YOUR_ID", colab_url="",
                 leaderboard_rmse=None, ta_rmse=None, student_name=""):
    """Build a report from actual output files, retaining all incomplete-status flags.

    Under the TA clarification dated 2026-10-19, the starter and plotting helper
    are optional references. A Colab link and leaderboard score cannot be inferred
    from a local experiment; those items remain pending unless supplied/verified.
    """
    root = Path(results_dir).resolve()
    path = root / "results.json"
    if not path.is_file():
        raise FileNotFoundError(
            "No results.json exists. Run the official-data experiment first. "
            "For the analytical solution only, use build_part1()."
        )
    results = json.loads(path.read_text(encoding="utf-8"))
    styles = _styles()
    metadata = results.get("metadata", {})
    baseline = results.get("baseline", {})
    official_reference = results.get("official_baseline_reference", OFFICIAL_BASELINE_REFERENCE)
    if not isinstance(official_reference, dict):
        official_reference = OFFICIAL_BASELINE_REFERENCE
    selection = results.get("selection", {})
    curves = results.get("learning_curves", {})
    cfg = results.get("configuration", {})
    helper_status = curves.get("helper_status")
    if isinstance(helper_status, dict):
        state = helper_status.get("state")
        # Translate old runner statuses using the later TA clarification, rather
        # than reintroducing an obsolete mandatory-helper instruction from JSON.
        if state in {"optional_not_used", "not_provided", None}:
            helper_text = "Optional reference not used; independently implemented learning curves are permitted."
        elif state in {"optional_skipped", "integration_pending"}:
            helper_text = "Optional reference helper skipped; independently implemented learning curves are used."
        elif state in {"optional_called", "called_review_required"}:
            helper_text = "Optional reference helper was called; supplementary outputs should be inspected for consistency."
        elif state == "optional_partial_or_failed":
            helper_text = "Some optional helper calls were skipped or failed; independent learning curves remain available. Call details are recorded in results.json."
        else:
            helper_text = "Optional reference helper status: " + str(state) + ". See results.json for call details."
    else:
        helper_text = "No optional helper use recorded; independently implemented learning curves are permitted."
    leaderboard = results.get("leaderboard", {})
    if not student_id or student_id == "YOUR_ID":
        student_id = str(metadata.get("student_id") or "YOUR_ID")
    student_name = student_name or str(metadata.get("student_name") or "")
    configured_team = (f"{student_id}_{student_name}"
                       if student_id != "YOUR_ID" and student_name else "")
    team_name = configured_team or leaderboard.get("team_name") or ""
    test_rmse = leaderboard_rmse if leaderboard_rmse is not None else leaderboard.get("test_rmse")
    ta_score = ta_rmse if ta_rmse is not None else leaderboard.get("ta_baseline_rmse")
    if ta_score is None:
        ta_score = official_reference.get("test_rmse")
    evidence = leaderboard.get("evidence", {})
    screenshot_source = isinstance(evidence, dict) and evidence.get("type") == "user_uploaded_screenshot"
    team_verified = screenshot_source and bool(team_name) and team_name == leaderboard.get("team_name")
    score_verified = (screenshot_source and test_rmse is not None
                      and test_rmse == leaderboard.get("test_rmse"))
    rank_in_snapshot = leaderboard.get("rank_in_screenshot") if team_verified and score_verified else None
    score_source_text = ("recorded from the user's Kaggle screenshot" if score_verified
                         else "supplied leaderboard result")
    s = []

    def p(text, kind="A2Body", raw=False):
        s.append(_p(text, styles, kind, raw=raw))
    def section(label, title):
        s.append(PageBreak())
        p(label, "A2Label")
        p(title, "A2H1")
    def figure(filename, caption, max_height=285):
        fp = (root / filename).resolve()
        if not fp.is_relative_to(root):
            raise ValueError("Report figure path escapes results directory.")
        if fp.is_file():
            s.append(KeepTogether([
                _chart_image(fp, max_height=max_height),
                _p(caption, styles, "A2Caption"),
            ]))
        else:
            p("Required figure is missing: " + str(filename), "A2Small")

    p("EMPIRICAL REPORT  /  VERIFY PENDING ITEMS BEFORE SUBMISSION", "A2Label")
    p("The Morning Rush", "A2Title")
    p("Assignment 2: linear models for hourly YouBike demand", "A2H2")
    p(f"Student ID: {student_id}" + (f"    Name: {student_name}" if student_name else ""))
    p(f"Experiment timestamp: {metadata.get('generated_at', 'not recorded')}")
    p(f"Random seed: {metadata.get('seed', 0)}. scikit-learn: {metadata.get('sklearn_version', 'not recorded')}.")
    status_text = "<b>Report status:</b> measured experiments are loaded from results.json. "
    if test_rmse is None:
        status_text += "The model's Kaggle result remains pending until an actual submission is scored. "
    else:
        status_text += f"Kaggle test RMSE {_num(test_rmse, 5)} is {score_source_text}. "
        if ta_score is not None:
            status_text += ("It is strictly below " if float(test_rmse) < float(ta_score)
                            else "It does not beat ") + f"the official TA threshold {_num(ta_score)}. "
    status_text += "The starter and plotting helper are optional (TA, 19 October 2026). "
    pending_items = []
    if student_id == "YOUR_ID" or not student_name:
        pending_items.append("student details")
    if not colab_url:
        pending_items.append("a viewable Colab link")
    if test_rmse is None:
        pending_items.append("the actual leaderboard score")
    status_text += ("Still required: " + ", ".join(pending_items) + "."
                    if pending_items else "The required identity, link and leaderboard fields are supplied.")
    s.append(_callout(status_text, styles))
    s.append(Spacer(1, 12))
    s.append(_table(["Implementation / submission item", "Recorded status"], [
        ["Starter notebook", "Optional reference; an independently written notebook is permitted (TA, 2026-10-19)."],
        ["Optional plotting helper", helper_text],
        ["Viewable Colab notebook", colab_url or "Pending: provide a viewable Colab URL"],
        ["Kaggle Team Name", (team_name + (" (verified in the supplied screenshot)" if team_verified else ""))
         if team_name else "Required format: StudentID_Name. Enter the actual student ID and name on Kaggle."],
        ["Leaderboard test RMSE", _num(test_rmse, 5) if test_rmse is not None else "Pending leaderboard submission"],
        ["TA test baseline RMSE", (_num(ta_score) + " (official baseline.txt)") if ta_score is not None else "Official reference not recorded"],
    ], styles, [182, CONTENT_W-182]))
    s.append(Spacer(1, 12))
    p("Evaluation protocol", "A2H2")
    p("January-April 2026 is the development training period; May 2026 is the fixed "
      "validation month. June 2026 is the unlabeled test period. Every learned "
      "preprocessing transform is fitted inside its training pipeline. Final model "
      "ranking uses May only within the explicitly eligible stable candidate set; "
      "the selected pipeline is refitted on January-May "
      "before producing June predictions.")
    p("Metrics are measured in the original rentals scale. The baseline and feature "
      "comparison report unmodified predictions; final-model selection additionally "
      "reports the effect of clipping negative predictions to zero. A log-target "
      "model is inverse-transformed before scoring.")
    p("The explicit data specification takes precedence over the handout's inconsistent "
      "references to 'one month' and 'the final week of March'.")
    if metadata.get("quick_mode"):
        p("This result file is marked as a quick program check. It must not be treated as the complete formal experiment.", "A2Small")

    with tempfile.TemporaryDirectory(prefix="a2_report_") as directory:
        s.append(PageBreak())
        s.extend(_part1_story(styles, directory, student_id=student_id, standalone=False))

        section("PART II(a)", "Data exploration and the baseline")
        summary = results.get("data_summary", {})
        s.append(_table(["Dataset item", "Observed value"], [
            ["All labeled rows (January-May)", summary.get("train_rows")],
            ["Development training rows (January-April)", summary.get("fit_rows")],
            ["Validation rows (May)", summary.get("validation_rows")],
            ["Test rows (June)", summary.get("test_rows")],
            ["Station count", summary.get("station_count")],
        ], styles, [CONTENT_W*.66, CONTENT_W*.34]))
        s.append(Spacer(1, 12))
        p("The baseline uses only temperature, precipitation, humidity, and wind speed "
          "with LinearRegression. Because weather is shared across stations in an "
          "hour, this baseline cannot assign different predictions to those stations "
          "at the same timestamp.")
        s.append(_table(["Source", "Train RMSE", "Train R²", "May RMSE", "May R²"], [
            ["This run: weather OLS", _num(baseline.get("train_rmse")), _num(baseline.get("train_r2")),
             _num(baseline.get("val_rmse")), _num(baseline.get("val_r2"))],
            ["Official baseline.txt", _num(official_reference.get("train_rmse")), _num(official_reference.get("train_r2")),
             _num(official_reference.get("val_rmse")), _num(official_reference.get("val_r2"))],
        ], styles, [139, 91, 84, 106, CONTENT_W-420]))
        s.append(Spacer(1, 8))
        p("The second row quotes the supplied official reference; the first row is independently "
          "computed on the supplied CSV files. The official TA test RMSE is "
          + _num(official_reference.get("test_rmse")) + ", which is a leaderboard threshold, "
          "not this model's measured June performance.")
        computed_r2, reference_r2 = baseline.get("val_r2"), official_reference.get("val_r2")
        if isinstance(computed_r2, (int, float)) and isinstance(reference_r2, (int, float)):
            if not math.isclose(computed_r2, reference_r2, rel_tol=0, abs_tol=0.00005):
                p(f"Reference discrepancy: this run computes May R² = {computed_r2:.10f}, "
                  f"whereas baseline.txt lists {_num(reference_r2)}. The reference repeats "
                  "the same R² for training and validation; the cause of this inconsistency "
                  "cannot be established from the text alone. The computed R² is retained, "
                  "and the official test-RMSE threshold remains a separately sourced value.")
        s.append(Spacer(1, 12))
        p("Target transformation", "A2H2")
        p("The raw target is compared with log(1 + rentals). Log predictions are "
          "converted using exp(prediction) − 1 before computing the following metrics. "
          "This permits a comparison on the leaderboard's original rental-count scale.")
        rows = results.get("target_comparison", [])
        if rows:
            s.append(_table(["Model / target", "Train RMSE", "Train R²", "May RMSE", "May R²"], [[
                f"{r.get('model', 'LinearRegression')} / {r.get('target', '')}",
                _num(r.get("train_rmse")), _num(r.get("train_r2")),
                _num(r.get("val_rmse")), _num(r.get("val_r2")),
            ] for r in rows], styles, [139, 91, 84, 106, CONTENT_W-420]))
        finite_targets = [r for r in rows if isinstance(r.get("val_rmse"), (int,float)) and math.isfinite(r["val_rmse"])]
        if finite_targets:
            best = min(finite_targets, key=lambda r:r["val_rmse"])
            p(f"For this controlled target comparison, {best.get('target')} has the "
              f"lowest May RMSE ({_num(best['val_rmse'])}). Taking logs changes the "
              "weight given to large rental counts; an improved distribution shape "
              "alone does not guarantee a lower RMSE in the original units.")

        section("PART II(a)  /  EXPLORATORY EVIDENCE", "Patterns used to motivate features")
        eda = results.get("diagnostics",{}).get("eda",{})
        p(f"Across the development training data, the highest overall mean demand "
          f"occurs at hour {eda.get('peak_hour_overall', 'not recorded')}, and the lowest "
          f"at hour {eda.get('lowest_hour_overall', 'not recorded')}. The hourly curves "
          "and daily trend motivate explicit time and calendar features.")
        p(f"Station mean rentals range from {_num(eda.get('lowest_station_mean'))} "
          f"to {_num(eda.get('highest_station_mean'))} per station-hour. Station "
          "one-hot features can represent these level differences. The heatmap "
          "examines whether station-specific hourly interactions are also warranted.")
        p("These descriptive relationships motivate candidates; only the held-out "
          "May comparison establishes whether a candidate improves forecasting.")
        figure("figures/eda_hourly.png", "Hourly demand, separated by the calendar holiday flag.")
        figure("figures/eda_stations.png", "Station-level demand patterns in the development training data.")
        section("PART II(a)  /  WEATHER AND TARGET", "Weather and rental-count distributions")
        p(f"Rental counts have mean {_num(eda.get('mean_rentals'))}, median "
          f"{_num(eda.get('median_rentals'))}, maximum {_num(eda.get('max_rentals'),0)}, "
          f"and sample skewness {_num(eda.get('target_skewness'))}. The zero-count "
          f"fraction is {_num(eda.get('zero_fraction'))}. The raw/log comparison "
          "therefore evaluates a distribution-motivated option on the actual RMSE objective.")
        rain_groups = eda.get("rain_groups", [])
        if rain_groups:
            s.append(_table(["Precipitation group (mm)", "Mean rentals", "Station-hour rows"], [
                [r.get("rain_mm"), _num(r.get("mean_rentals")), r.get("count")]
                for r in rain_groups], styles, [CONTENT_W*.44, CONTENT_W*.27, CONTENT_W*.29]))
            s.append(Spacer(1, 8))
        figure("figures/eda_weather.png", "Weather relationships estimated only from the development training period.")
        figure("figures/eda_target.png", "Rental-count distribution and its log transformation.")
        p("These exploratory plots identify associations, not causal effects. "
          "Their feature hypotheses are evaluated on May rather than assumed to improve prediction.")

        section("PART II(b)", "Cumulative feature engineering")
        descriptions = cfg.get("feature_descriptions", {})
        compared_sets = list(dict.fromkeys(r.get("feature_set") for r in results.get("feature_comparison", [])
                                         if r.get("feature_set")))
        s.append(_table(["Set", "Cumulative feature design used in this run"], [
            [key, descriptions.get(key, "See the recorded pipeline configuration.")]
            for key in compared_sets
        ], styles, [48, CONTENT_W-48]))
        s.append(Spacer(1, 10))
        p("Categorical hours, weekdays and stations are one-hot encoded. This avoids "
          "imposing an artificial linear order on station identity or forcing demand "
          "to increase monotonically from hour 0 to hour 23. Interactions allow the "
          "hourly demand profile to vary, instead of shifting the same profile up or down.")
        p("Each feature set uses the same Ridge configuration and preprocessing/target "
          "choice. Feature count is measured after encoding and expansion. The "
          "four-weather OLS baseline is reported separately, so model changes are "
          "not attributed to feature additions.")
        p(f"Controlled configuration: Ridge alpha = {cfg.get('feature_comparison_alpha', 'not recorded')}; "
          f"numeric scaler = {cfg.get('feature_comparison_scaler', 'not recorded')}; "
          f"target = {results.get('target_decision', {}).get('selected', 'not recorded')}.")
        rows = results.get("feature_comparison", [])
        s.append(_table(["Set", "Features", "Train RMSE", "May RMSE", "May R²"], [[
            r.get("feature_set"), r.get("n_features"), _num(r.get("train_rmse")),
            _num(r.get("val_rmse")), _num(r.get("val_r2")),
        ] for r in rows], styles, [47, 82, 127, 127, CONTENT_W-383]))
        s.append(Spacer(1, 9))
        gains = results.get("diagnostics",{}).get("feature_gains",[])
        if gains:
            p("Observed gains", "A2H2")
            s.append(_table(["Added feature stage", "May RMSE reduction", "Improved?"], [
                [f"{r.get('from')} to {r.get('to')}", _num(r.get("validation_rmse_reduction")),
                 "Yes" if r.get("improved") else "No"] for r in gains], styles,
                [CONTENT_W*.45, CONTENT_W*.35, CONTENT_W*.20]))
            finite_gains = [r for r in gains if isinstance(r.get('validation_rmse_reduction'), (int,float))]
            if finite_gains:
                best_gain = max(finite_gains, key=lambda r:r['validation_rmse_reduction'])
                delta = best_gain['validation_rmse_reduction']
                if delta > 0:
                    p(f"The largest marginal improvement is {best_gain.get('from')} to "
                      f"{best_gain.get('to')}, reducing May RMSE by {_num(delta)}. "
                      "The corresponding added feature family supplies information that "
                      "the preceding representation could not express as effectively.")
                else:
                    p("No cumulative addition reduced May RMSE in this run. The proposed "
                      "features therefore are not supported as improvements by this validation result.")

        section("PART II(c)", "Feature scaling: controlled comparisons")
        p("Numeric preprocessing is compared across no scaling, standardization, "
          "and min-max scaling. Categorical encoding remains fixed. LinearRegression "
          "represents least squares, Ridge supplies regularization, and SGDRegressor "
          "uses iterative gradient updates. SGD hyperparameters are held fixed across scalers.")
        p(f"The scaling experiment fixes feature set {cfg.get('scaling_feature_set', 'not recorded')} "
          f"and target {cfg.get('scaling_target', 'not recorded')}. Ridge alpha is 10. "
          "The OLS and SGD models have no regularization in this comparison.")
        rows = results.get("scaling_comparison", [])
        s.append(_table(["Model", "Numeric scaler", "May RMSE", "Epochs", "Status"], [[
            r.get("model"), r.get("scaler"), _num(r.get("val_rmse")),
            str(r.get("epochs")) if r.get("epochs") is not None else "N/A",
            r.get("status", "not recorded"),
        ] for r in rows], styles, [117, 106, 103, 59, CONTENT_W-385]))
        s.append(Spacer(1, 10))
        p("Epochs refer to SGD's full passes over the training data, not individual "
          "sample updates. A missing RMSE caused by failure/nonfinite predictions "
          "is retained as N/A, with the failure status; it is not replaced by a score.")
        p("Part I explains the mechanisms: under equivalent, numerically well-behaved "
          "least-squares representations, invertible feature rescaling preserves "
          "the prediction space; fixed-alpha Ridge changes its effective penalty; "
          "SGD also experiences a changed optimization geometry. Scaling need not "
          "improve every validation score.")
        p("Observed behavior and optimizer settings", "A2H2")
        scale_diag = results.get("diagnostics",{}).get("scaling",{})
        settings = scale_diag.get("sgd_settings",{})
        if settings:
            p("SGD settings: " + "; ".join(f"{k}={v}" for k,v in settings.items()) + ".", "A2Small")
        for model in ["LinearRegression", "Ridge", "SGDRegressor"]:
            finite = [r for r in rows if r.get('model') == model and
                      isinstance(r.get('val_rmse'), (int,float)) and math.isfinite(r['val_rmse'])]
            if finite:
                best_scaler = min(finite, key=lambda r:r['val_rmse'])
                worst_scaler = max(finite, key=lambda r:r['val_rmse'])
                p(f"{model}: finite May RMSE ranges from {_num(best_scaler['val_rmse'])} "
                  f"({best_scaler.get('scaler')}) to {_num(worst_scaler['val_rmse'])} "
                  f"({worst_scaler.get('scaler')}). Differences reflect the mechanisms "
                  "above under the fixed settings, rather than a guarantee that one scaler always wins.")
        warnings_seen = []
        for r in rows:
            warning = r.get("warnings")
            if warning:
                if isinstance(warning, list): warning = "; ".join(str(x) for x in warning)
                warnings_seen.append(f"{r.get('model')} / {r.get('scaler')}: {warning}")
        if warnings_seen:
            p("Recorded convergence warnings", "A2H2")
            for warning in warnings_seen:
                p(warning, "A2Small")

        section("PART III(a)-(b)", "Learning curves and remedies")
        p(str(curves.get("cv_description", "Cross-validation configuration was not recorded.")))
        if curves.get("regularization_selection_note"):
            p(str(curves["regularization_selection_note"]), "A2Small")
        models = curves.get("models", {})
        if curves.get("remedy_description") and not models.get("B_after", {}).get("description"):
            s.extend(_interpretation_paragraphs(curves["remedy_description"], styles, "Remedy design"))
        p("The horizontal axis is training-set size, not epochs. Before/after "
          "comparisons use the same temporal boundaries and original-scale RMSE.")
        for model_name in ["A_before", "B_before", "A_after", "B_after"]:
            if models.get(model_name,{}).get("description"):
                p(model_name.replace("_", " ") + ": " + models[model_name]["description"])
        curve_rows = []
        for name in ["A_before", "B_before", "A_after", "B_after"]:
            info = models.get(name, {})
            summary_model = info.get("summary", {})
            curve_rows.append([
                name.replace("_", " "), _num(summary_model.get("train_rmse")),
                _num(summary_model.get("val_rmse", summary_model.get("validation_rmse"))),
                _num(summary_model.get("gap")),
            ])
        reference_summary = models.get("A_before", {}).get("summary", {})
        size_rows = reference_summary.get("max_training_examples_mean")
        size_hours = reference_summary.get("max_training_hours")
        fold_count = len(curves.get("folds", []))
        if size_rows is not None and size_hours is not None:
            fold_label = f"{fold_count}-fold means" if fold_count else "means across the recorded folds"
            p(f"Table values are {fold_label} at the largest common training size: "
              f"{_num(size_rows, 0)} station-hour rows ({_num(size_hours, 0)} complete hours) "
              "per fold. These are January-April temporal-CV measurements, not May "
              "holdout scores or errors from the final January-May refit.", "A2Small")
        s.append(_table(["Model", "Train RMSE", "Temporal CV RMSE", "Gap"], curve_rows,
                        styles, [137, 135, 135, CONTENT_W-407]))
        s.append(Spacer(1, 10))
        a_before = models.get("A_before", {}).get("summary", {})
        a_after = models.get("A_after", {}).get("summary", {})
        b_before = models.get("B_before", {}).get("summary", {})
        b_after = models.get("B_after", {}).get("summary", {})
        if all(summary_model.get(key) is not None
               for summary_model in [a_before, a_after, b_before, b_after]
               for key in ["train_rmse", "val_rmse", "gap"]):
            p("Model A: original train/CV RMSE is " + _num(a_before["train_rmse"]) + "/"
              + _num(a_before["val_rmse"]) + " (gap " + _num(a_before["gap"]) + "). "
              "Richer features reduce CV RMSE to " + _num(a_after["val_rmse"])
              + " and the gap to " + _num(a_after["gap"]) + ". The relatively small "
              "original gap, high errors and clear improvement from richer features "
              "support a high-bias diagnosis for the original representation.")
            p("Model B: original train/CV RMSE is " + _num(b_before["train_rmse"]) + "/"
              + _num(b_before["val_rmse"]) + " (gap " + _num(b_before["gap"]) + "), "
              "showing severe variance/extrapolation instability. The remedy lowers CV "
              "RMSE to " + _num(b_after["val_rmse"]) + " and the gap to "
              + _num(b_after["gap"]) + ". Lower polynomial degree, training-only weather "
              "bounds and Ridge jointly stabilize the model; the improvement cannot "
              "be attributed to regularization alone.")
        else:
            s.extend(_interpretation_paragraphs(
                results.get("diagnostics",{}).get("learning_curve_interpretations",{}), styles))
        section("PART III  /  LEARNING CURVE FIGURES", "Before and after the remedies")
        if curves.get("figure_axis_note"):
            p(str(curves["figure_axis_note"]), "A2Small")
        figure("figures/learning_before.png", "Before remedies: the simple and flexible models under the same CV protocol.", 280)
        figure("figures/learning_after.png", "After remedies: richer features for A and the recorded stabilization remedy for B.", 280)
        a_points = models.get("A_before", {}).get("points", [])
        b_points = models.get("B_before", {}).get("points", [])
        if len(a_points) >= 2:
            p("More data for A: the final two original CV errors are "
              + _num(a_points[-2].get("validation_rmse")) + " to "
              + _num(a_points[-1].get("validation_rmse"))
              + ". This plateau indicates limited benefit from more data of the same "
              "kind; improving the representation is the effective remedy here.", "A2Small")
        if len(b_points) >= 2:
            rises = [(left.get("validation_rmse"), right.get("validation_rmse"))
                     for left, right in zip(b_points[:-1], b_points[1:])
                     if left.get("validation_rmse") is not None
                     and right.get("validation_rmse") is not None
                     and right["validation_rmse"] > left["validation_rmse"]]
            text = ("More data for B: original CV RMSE falls from "
                    + _num(b_points[0].get("validation_rmse")) + " to "
                    + _num(b_points[-1].get("validation_rmse")))
            if rises:
                text += ", but nonmonotonically (" + _num(rises[0][0]) + " to " + _num(rises[0][1]) + " mid-curve)"
            p(text + ". More representative data may help, but the remaining large "
              "error and gap show that more data alone is insufficient without the "
              "stabilizing remedy.", "A2Small")

        official_figures = {k:v for k,v in results.get("files",{}).get("figures",{}).items()
                            if str(k).startswith("official_")}
        if official_figures:
            section("PART III  /  OPTIONAL REFERENCE OUTPUT", "Supplementary course-helper plots")
            p("These plots were produced by calls to the provided course helper. "
              "Verify their score sign, metric, split, and training-size behavior "
              "when interpreting them. Under the TA clarification, this helper is "
              "optional; the independently implemented temporal learning curves above "
              "are the primary analysis in this report.")
            for stage in ["before", "after"]:
                images = []
                labels = []
                for prefix in ["A", "B"]:
                    key = f"official_{prefix}_{stage}"
                    relative = official_figures.get(key)
                    if relative:
                        fp = (root / relative).resolve()
                        if not fp.is_relative_to(root):
                            raise ValueError("Official helper figure escapes results directory.")
                        if fp.is_file():
                            im = _chart_image(fp, max_height=260)
                            factor = min(1, ((CONTENT_W-16)/2)/im.drawWidth)
                            im.drawWidth *= factor
                            im.drawHeight *= factor
                            images.append(im)
                            labels.append(f"{prefix} {stage}")
                if images:
                    p(" / ".join(labels), "A2H2")
                    width = CONTENT_W / len(images)
                    s.append(Table([images], colWidths=[width]*len(images), hAlign="CENTER"))
                    s.append(Spacer(1, 9))

        section("PART III(c)", "Selected model and May validation")
        s.append(_table(["Selected model item", "Value"], [
            ["Model", selection.get("model_name")], ["Feature set", selection.get("feature_set")],
            ["Ridge alpha, if applicable", selection.get("alpha") if selection.get("alpha") is not None else "Not applicable"],
            ["Target", selection.get("target")],
            ["May RMSE before clipping", _num(selection.get("validation_rmse_raw"))],
            ["May RMSE after clipping", _num(selection.get("validation_rmse_clipped"))],
            ["May R² after clipping", _num(selection.get("validation_r2_clipped"))],
        ], styles, [CONTENT_W*.57, CONTENT_W*.43]))
        s.append(Spacer(1, 11))
        selection_policy = (cfg.get("final_model_selection_policy")
                            or selection.get("eligibility_rule")
                            or results.get("selection_policy"))
        if selection_policy:
            s.extend(_interpretation_paragraphs(selection_policy, styles, "Final-selection eligibility"))
        selected_description = cfg.get("feature_descriptions", {}).get(selection.get("feature_set"))
        if selected_description:
            p("Selected representation: " + str(selected_description))
        figure("figures/validation_predictions.png",
               "May predictions from the January-April fit, before the final January-May refit.", 252)
        p("The daily series compares average station-hour demand, while the scatter "
          "shows individual May observations. Deviations from the diagonal expose "
          "errors that can be hidden by daily averages. These are held-out May "
          "predictions; they provide no direct measurement of June test performance.")

        section("PART III(c)  /  FINAL REFIT", "June submission and leaderboard")
        p("The selected pipeline is refitted on all labeled January-May rows. June "
          "predictions are clipped below at zero, are not rounded to integers, "
          "and are aligned by the template's ID field. Station capacity is "
          "not imposed as an upper bound on hourly rentals.")
        if cfg.get("weather_bound_policy"):
            s.extend(_interpretation_paragraphs(cfg["weather_bound_policy"], styles, "Weather-range preprocessing"))
        submission_summary = results.get("submission_summary", {})
        id_column = submission_summary.get("id_column", summary.get("submission_id_column", "Not recorded"))
        prediction_column = submission_summary.get("prediction_column", summary.get("prediction_column", "Not recorded"))
        s.append(_table(["Submission check", "Recorded value"], [
            ["Filename", results.get("files", {}).get("submission", "submission.csv")],
            ["Final training rows (January-May)", submission_summary.get("training_rows", summary.get("train_rows"))],
            ["Prediction rows (June)", submission_summary.get("rows", summary.get("test_rows"))],
            ["ID column / prediction column", str(id_column) + " / " + str(prediction_column)],
            ["Prediction range", _num(submission_summary.get("min_prediction")) + " to "
             + _num(submission_summary.get("max_prediction"))],
            ["Negative predictions", submission_summary.get("negative_predictions")],
            ["ID order matches the sample", "Yes" if submission_summary.get("id_order_matches_sample") is True
             else str(submission_summary.get("id_order_matches_sample", "Not recorded"))],
        ], styles, [CONTENT_W*.57, CONTENT_W*.43]))
        s.append(Spacer(1, 10))
        p("The actual sample_submission.csv determines header casing, column order "
          "and ID row order. The competition-page example ID,Rentals does not "
          "override the downloaded template.")
        p("Leaderboard requirement: test RMSE must be strictly below the official TA "
          "baseline (equality does not qualify). Official threshold: " + _num(ta_score) + ".")
        if test_rmse is None:
            p("Leaderboard result: PENDING. The test CSV does not contain rental "
              "labels, so official test RMSE cannot be computed locally. Upload "
              "submission.csv and record the returned score before finalizing this section.")
        else:
            p("Recorded leaderboard test RMSE: " + _num(test_rmse, 5) + " ("
              + score_source_text + ").")
            if rank_in_snapshot is not None:
                entries = leaderboard.get("entries_in_screenshot")
                snapshot_text = f"The supplied screenshot shows team {team_name} at rank {rank_in_snapshot}"
                if entries is not None:
                    snapshot_text += f" with {entries} recorded submission" + ("s" if entries != 1 else "")
                snapshot_text += ". This is the screenshot's ranking, not a live rank."
                if leaderboard.get("evaluation_scope") == "all_test_data_as_stated_in_screenshot":
                    snapshot_text += " Its banner states that the leaderboard uses all test data."
                p(snapshot_text)
            if ta_score is None:
                p("TA baseline comparison remains pending: enter the official TA test RMSE.")
            else:
                wins = float(test_rmse) < float(ta_score)
                comparison = (f"{_num(test_rmse, 5)} < {_num(ta_score)}: the recorded score meets "
                              "the strict leaderboard-baseline requirement."
                              if wins else "The recorded score does not beat the official TA baseline.")
                if wins and float(ta_score) > 0:
                    comparison += f" RMSE is {100 * (1 - float(test_rmse) / float(ta_score)):.2f}% lower."
                p(comparison)
            val = selection.get("validation_rmse_clipped")
            if isinstance(val, (int,float)) and math.isfinite(val):
                diff = float(test_rmse) - float(val)
                comparison = f"May validation RMSE {_num(val)} and observed June RMSE {_num(test_rmse, 5)} "
                if diff < 0 and val > 0:
                    comparison += (f"are reasonably close: June is {-diff:.4f} rentals "
                                   f"({-100 * diff / float(val):.2f}% of May RMSE) lower. "
                                   "May was therefore conservative for this observed June result. ")
                else:
                    comparison += f"differ by {diff:+.4f} rentals (June minus May). "
                p(comparison + "May evaluates a January-April fit; June evaluates the final "
                  "January-May refit. Both the period and training-data size change, "
                  "so the gap does not isolate a causal explanation.")
        p(("Verified team naming: " + str(team_name) + " follows StudentID_Name. "
           if team_verified else "Set the Kaggle Team Name to StudentID_Name. ")
          + "A name correction alone does not require another prediction upload (TA announcement).",
          "A2Small")

        tuning = results.get("tuning", [])
        if tuning:
            section("APPENDIX  /  MODEL SELECTION RECORD", "May-only candidate comparison")
            p("The following candidate scores were produced before June labels were "
              "available. Final ranking minimizes clipped May RMSE only among rows "
              "marked eligible. The original unbounded degree-4 B is retained as a "
              "diagnostic row, with its exclusion visible. No leaderboard feedback is used.")
            s.append(_table(["Candidate", "Set", "Alpha", "Target", "Eligible", "Clipped May RMSE"], [[
                r.get("model_name"), r.get("feature_set"),
                f"{float(r['alpha']):g}" if r.get("alpha") is not None else "N/A",
                r.get("target"),
                "Yes" if r.get("eligible_for_final_selection") is True else
                ("No: diagnostic" if r.get("eligible_for_final_selection") is False else "Not recorded"),
                _num(r.get("val_rmse_clipped")),
            ] for r in tuning], styles, [155, 54, 51, 61, 67, CONTENT_W-388]))
            excluded = [r for r in tuning if r.get("eligible_for_final_selection") is False]
            for row in excluded:
                p(str(row.get("model_name")) + " exclusion: "
                  + str(row.get("eligibility_reason") or row.get("exclusion_reason")
                        or "See the recorded selection policy."), "A2Small")
        development_note = results.get("development_notes") or results.get("diagnostics", {}).get("development_note")
        if isinstance(development_note, dict):
            # Detailed curve values are already in Part III; retain the rationale here.
            development_note = {k: v for k, v in development_note.items()
                                if k not in {"training_curve_evidence", "eligibility_rule"}}
        if development_note:
            p("Development rationale", "A2H2")
            s.extend(_interpretation_paragraphs(development_note, styles))

        section("COMPLETION CHECK", "Remaining verification and sources")
        p(("Remaining item: " + ", ".join(pending_items) + "."
           if pending_items else "The required identity, Colab link and leaderboard fields are supplied."))
        p("The TA announcement of 19 October 2026 permits independent implementations; "
          "the starter and plotting helper are optional. Retain executed notebook "
          "outputs, required analyses, tables and learning curves. The report must "
          "include a viewable Colab link.")
        p(("The supplied screenshot verifies student/team " + str(team_name) + ". "
           if team_verified else "Use StudentID_Name as the Kaggle Team Name. ")
          + "Resubmission is not required solely to correct a team name.")
        p("The announced deadline is 6 November 2026 at 23:59, with at most five "
          "submissions per day. The updated late penalties are 25% for 1 minute-1 day, "
          "50% for 1-2 days, and 100% after 2 days (course announcement time).")
        limitations = []
        for item in results.get("limitations", []):
            text = str(item)
            # These three notices came from the original handout interpretation;
            # the later TA clarification explicitly supersedes them.
            if "this standalone runner cannot verify that requirement" in text:
                continue
            if "integrate and run the official helper before submission" in text:
                continue
            if "starter integration still require inspection" in text:
                continue
            limitations.append(item)
        s.extend(_interpretation_paragraphs(limitations, styles, "Recorded limitations"))
        p("Sources", "A2H2")
        p("1. A2_Handout.pdf, pages 1-4: task, toy dataset, constraints, split and deliverables.")
        p("2. TA announcement, Additional Notes for Assignment 2, 19 October 2026: "
          "starter/reference-helper clarification and Kaggle naming instructions. "
          "This later clarification supersedes the handout's starter/helper wording.")
        p(TA_NOTE_URL, "A2Small")
        p("3. Supplied A2_data_and_baseline_1_6 dataset: train.csv, test.csv and "
          "sample_submission.csv are the empirical inputs. Input SHA-256 values and "
          "the experiment metadata are retained in results.json.")
        p("4. The supplied baseline.txt: published weather-only baseline metrics and "
          "the official TA test-RMSE threshold of 24.2461. Its stated May R² is "
          "distinguished from this report's recomputed R².")
        p("5. User-supplied Kaggle competition description and course submission "
          "announcement: data dictionary, sample-format precedence, strict baseline "
          "comparison, deadline, daily limit and updated late penalties.")
        p(KAGGLE_INVITE_URL, "A2Small")
        p("6. scikit-learn documentation: Pipeline; LinearRegression; Ridge; "
          "SGDRegressor; StandardScaler; TransformedTargetRegressor; learning_curve.")
        p("Documentation: https://scikit-learn.org/stable/", "A2Small")
        if score_verified:
            p("7. User-supplied Kaggle leaderboard screenshot, "
              + str(evidence.get("source_filename", "uploaded image"))
              + ": verifies the displayed team, score and snapshot ranking. "
              "The screenshot states that all test data are used; it is not a live ranking query.")
        p("Empirical tables and figures come from results.json and its outputs. "
          "Leaderboard scores are supplied separately, never inferred from unlabeled test data.")
        _document(output_pdf, "A2 - YouBike regression report",
                  author=" ".join(value for value in [student_id, student_name] if value)).build(s)
    return str(Path(output_pdf).resolve())


def _main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--part1", action="store_true")
    parser.add_argument("--results", default="a2_results")
    parser.add_argument("--output", required=True)
    parser.add_argument("--student-id", default="YOUR_ID")
    parser.add_argument("--student-name", default="")
    parser.add_argument("--colab-url", default="")
    parser.add_argument("--leaderboard-rmse", type=float)
    parser.add_argument("--ta-rmse", type=float)
    args = parser.parse_args()
    if args.part1:
        path = build_part1(args.output, args.student_id)
    else:
        path = build_report(args.results, args.output, args.student_id, args.colab_url,
                            args.leaderboard_rmse, args.ta_rmse, args.student_name)
    print(path)


if __name__ == "__main__":
    _main()


Writing a2_report.py


## Run the official-data experiments

The function below reports missing or malformed files rather than generating
replacement data. The built-in learning curves implement the required
training-size / RMSE analysis. The official helper is an optional reference;
if supplied, its invocation and output are recorded separately.

By default, the packaged `a2_results/results.json` is reused only when its
three input-file hashes and `pipeline_source_sha256` match the current files,
its full-run flag is recorded, and all referenced outputs exist. Reuse displays
the recorded experiment, including its original software versions and timestamp;
it does not claim that model fitting was repeated during this notebook execution.
Set `REUSE_SAVED_RESULTS=False` to recompute all models and learning curves.


In [ ]:
import importlib
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
import a2_pipeline
importlib.reload(a2_pipeline)

official_helper = globals().get('plot_learning_curve')
if not callable(official_helper):
    official_helper = None
    print('Using the built-in temporal learning curves; the course helper is optional.')

results = None
if DATA_READY and RUN_OFFICIAL_EXPERIMENTS:
    def file_sha256(path):
        digest = hashlib.sha256()
        with Path(path).open('rb') as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b''):
                digest.update(block)
        return digest.hexdigest()

    input_hashes = {name: file_sha256(DATA_DIR / name) for name in REQUIRED_CSV}
    pipeline_hash = file_sha256(Path(a2_pipeline.__file__).resolve())
    saved_results_path = OUTPUT_DIR / 'results.json'
    if REUSE_SAVED_RESULTS and saved_results_path.is_file() and not QUICK_CHECK_ONLY:
        try:
            candidate = json.loads(saved_results_path.read_text(encoding='utf-8'))
            candidate_meta = candidate.get('metadata', {})
            saved_files = candidate.get('files', {})
            required_outputs = [saved_files.get('submission', 'submission.csv')]
            required_outputs += list(saved_files.get('figures', {}).values())
            required_outputs += list(saved_files.get('tables', {}).values())
            output_root = OUTPUT_DIR.resolve()
            outputs_present = bool(required_outputs) and all(
                isinstance(relative, str)
                and (OUTPUT_DIR / relative).resolve().is_relative_to(output_root)
                and (OUTPUT_DIR / relative).is_file()
                for relative in required_outputs
            )
            required_sections = ['baseline', 'target_comparison', 'feature_comparison',
                                 'scaling_comparison', 'tuning', 'selection',
                                 'learning_curves', 'diagnostics', 'development_notes']
            measured_sections_present = all(candidate.get(key) for key in required_sections)
            recorded_policy = (
                candidate.get('configuration', {}).get('final_model_selection_policy')
                or candidate.get('selection', {}).get('eligibility_rule')
                or candidate.get('selection_policy')
            )
            curves_present = {'A_before', 'B_before', 'A_after', 'B_after'}.issubset(
                candidate.get('learning_curves', {}).get('models', {})
            )
            eligibility_recorded = all(isinstance(row.get('eligible_for_final_selection'), bool)
                                       for row in candidate.get('tuning', []))
            can_reuse = (
                candidate.get('data_summary', {}).get('input_sha256') == input_hashes
                and candidate_meta.get('pipeline_source_sha256') == pipeline_hash
                and candidate_meta.get('quick_mode') is False
                and candidate_meta.get('status') in {'experiments_complete_pending_leaderboard', 'experiments_complete'}
                and outputs_present
                and measured_sections_present and recorded_policy and curves_present
                and eligibility_recorded
            )
            if can_reuse:
                results = candidate
                print('Reusing saved official-data results after CSV / pipeline SHA-256 and output checks.')
                print('Recorded experiment timestamp:', candidate_meta.get('generated_at'))
                print('Recorded scikit-learn version:', candidate_meta.get('sklearn_version'))
                print('Model fitting was not repeated during this notebook execution.')
            else:
                print('Saved results do not match the current data/code or are incomplete; running the experiments.')
        except (OSError, ValueError, TypeError, AttributeError) as exc:
            print(f'Saved results could not be reused ({type(exc).__name__}); running the experiments.')
    if results is None:
        results = a2_pipeline.run_assignment(
            data_dir=DATA_DIR, output_dir=OUTPUT_DIR,
            student_id=STUDENT_ID or 'YOUR_ID',
            official_learning_curve=official_helper,
            quick=QUICK_CHECK_ONLY,
        )
        results.setdefault('metadata', {})['pipeline_source_sha256'] = pipeline_hash
        saved_results_path.write_text(
            json.dumps(results, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8'
        )
else:
    print('Part II–III not run: official CSV files are missing or execution is disabled.')


Using the built-in temporal learning curves; the course helper is optional.
Part II–III not run: official CSV files are missing or execution is disabled.


## Experiment tables

All RMSE and R² values below are computed in the original rentals unit.
The original weather-only baseline is retained. Cumulative feature comparisons
hold the model, target choice, and preprocessing fixed. Scaling comparisons
hold the feature set and model-specific optimization settings fixed.

The local training/May weather-only baseline below is distinct from the official
Kaggle test baseline. The supplied `baseline.txt` establishes the official test
threshold of **24.2461**; do not substitute a local validation score. Its published
May R² is compared with the computed value below; any discrepancy is disclosed,
while the computed result remains unchanged.


In [ ]:
if results is not None:
    from IPython.display import display, Markdown, Image
    display(Markdown('### Data and original baseline'))
    display(pd.DataFrame([results['baseline']]))
    reference_baseline = {'train_rmse': 25.0947, 'train_r2': 0.1437,
                          'val_rmse': 29.3875, 'val_r2': 0.1437}
    display(Markdown('### Baseline reproduction versus the supplied baseline.txt'))
    display(pd.DataFrame([
        {'metric': metric, 'computed': results['baseline'].get(metric),
         'official_file': reference_value}
        for metric, reference_value in reference_baseline.items()
    ]))
    computed_val_r2 = results['baseline'].get('val_r2')
    if computed_val_r2 is not None and not np.isclose(computed_val_r2, 0.1437, atol=5e-5, rtol=0):
        print(f'Official-file discrepancy: baseline.txt lists May R² = 0.1437; '
              f'the same weather-only OLS evaluation computes {computed_val_r2:.10f}.')
        print('The computed R² is retained. The repeated official train/May R² may be a transcription error.')
    print('Official TA test RMSE threshold from baseline.txt: 24.2461.')
    if LEADERBOARD_RMSE is None:
        print('The model test RMSE remains pending an actual scored Kaggle submission.')
    else:
        print(f'Recorded Kaggle test RMSE: {LEADERBOARD_RMSE:.5f}.')
        if TA_BASELINE_RMSE is not None:
            print('Strict official-baseline requirement met:', LEADERBOARD_RMSE < TA_BASELINE_RMSE)
    for title, key in [('Target transformation', 'target_comparison'),
                       ('Cumulative feature sets', 'feature_comparison'),
                       ('Scaling experiment', 'scaling_comparison'),
                       ('Validation-only model selection', 'tuning')]:
        display(Markdown('### '+title))
        display(pd.DataFrame(results[key]))
    print('Selected model:', results['selection'])
else:
    print('No experiment result is loaded in this execution. Check DATA_DIR and execution settings.')


No experiment result is loaded in this execution. Check DATA_DIR and execution settings.


## Plots and evidence-based analysis

The learning-curve x-axis is the number of training examples, not epochs.
For this saved official-data run, original Model A's final two CV RMSE values
are **27.6114 → 27.5959**, a plateau. More data of the same kind would have limited
benefit; richer features address the observed high-bias pattern and reduce the
largest-size CV RMSE to **16.4071**.

Original Model B's CV RMSE falls from **993.3459** at the smallest size to
**278.7592** at the largest, but the path is nonmonotonic: **570.6525 → 846.8590**
in the middle. More representative data may help, yet the remaining error and
large train/CV gap show that more data alone is insufficient. The combined
degree reduction, training-only weather bounds and Ridge remedy reduces CV
RMSE to **14.4042**; the gain cannot be assigned to regularization alone.


In [ ]:
if results is not None:
    import json
    seen_figures = set()
    for title, relative_path in results.get('files', {}).get('figures', {}).items():
        if isinstance(relative_path, str):
            figure_path = OUTPUT_DIR / relative_path
            if figure_path.is_file() and str(figure_path) not in seen_figures:
                display(Markdown('### '+title.replace('_', ' ').title()))
                display(Image(filename=str(figure_path)))
                seen_figures.add(str(figure_path))
    print(json.dumps(results.get('diagnostics', {}), ensure_ascii=False, indent=2))
    print('Helper status:', results.get('learning_curves', {}).get('helper_status'))
else:
    print('No experiment result is loaded in this execution; no empirical plots are displayed.')


No experiment result is loaded in this execution; no empirical plots are displayed.


## PDF report and submission file

The generated report includes the Part I derivation, real experiment tables and
figures, and measured diagnostics. Student ID `415511001`, name `嚴佳華`, team
`415511001_嚴佳華`, and test RMSE **12.37017** are recorded from the user's Kaggle
screenshot. Its displayed rank is **6** with **1** submission, and the page says
the leaderboard uses all test data. The rank is a snapshot, not a live query.
The report remains a draft because the viewable **Colab URL is still missing**.
The formal filenames are `A2_415511001.ipynb` and `A2_415511001.pdf`; using the
formal filename does not remove the outstanding Colab-link requirement.
The starter notebook is optional under the TA announcement of 19 October 2026.

The submitted Part III prediction file is `a2_results/submission.csv`. Access the competition through the
[class Kaggle invitation](https://www.kaggle.com/t/93a788c887e34269939b30da23305dfe).
The deadline is **6 November 2026 at 23:59** (course announcement time), and the
limit is **5 submissions per day**. The latest late-submission policy is 25% for
1 minute–1 day late, 50% for 1–2 days late, and 100% after 2 days.

The competition-page example uses `ID,Rentals`; the supplied official template
actually uses `id,rentals`. The downloaded `sample_submission.csv` is authoritative:
the generated submission follows its
exact column names and casing, column order, and ID row order. Do not force
lowercase headers. Hourly departures are not capped at station dock capacity.

`LEADERBOARD_RMSE=12.37017` is the actual score visible in the supplied screenshot.
If a changed prediction file is submitted later, record its corresponding score;
use `None` when the score is unknown.
`TA_BASELINE_RMSE=24.2461` has been verified against the supplied `baseline.txt`;
this is the TA baseline score, not this model's test result.
Full marks for the leaderboard component require **Kaggle test RMSE strictly
below the official TA baseline RMSE** (`LEADERBOARD_RMSE < TA_BASELINE_RMSE`);
an equal score does not meet this condition. The recorded comparison
**12.37017 < 24.2461** meets this requirement. A May validation score alone cannot
establish it. May RMSE **13.1623** was reasonably close and conservative for the
observed June RMSE **12.37017**: June was **0.7922** rentals, or **6.02%**, lower
than May (using May RMSE as the percentage denominator). May evaluates a
January–April fit, whereas the June submission uses the January–May refit;
both the evaluated month and training-data size differ, so the gap does not
isolate a causal explanation.
Rerun this section after filling the Colab URL. A local test RMSE cannot be
computed without June labels. Select the model on May, not by repeatedly tuning
on leaderboard feedback. Keep all notebook outputs when submitting.

The verified team name `415511001_嚴佳華` follows `StudentID_Name`. If a prediction
file was already submitted, changing only the team name does not require another submission.


In [ ]:
if results is not None:
    if importlib.util.find_spec('reportlab') is None:
        import subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'reportlab'])
    import a2_report
    importlib.reload(a2_report)
    report_path = OUTPUT_DIR / f'A2_{STUDENT_ID or "YOUR_ID"}.pdf'
    a2_report.build_report(
        results_dir=OUTPUT_DIR, output_pdf=report_path,
        student_id=STUDENT_ID or 'YOUR_ID', colab_url=COLAB_URL,
        leaderboard_rmse=LEADERBOARD_RMSE, ta_rmse=TA_BASELINE_RMSE,
        student_name=STUDENT_NAME,
    )
    print('Report:', report_path)
    print('Predictions:', OUTPUT_DIR / 'submission.csv')
    if not COLAB_URL:
        print('Remaining item: fill in a viewable Colab URL, then regenerate the report.')
    if LEADERBOARD_RMSE is None:
        print('Leaderboard score is pending; enter the actual scored result before submission.')
else:
    print('Full empirical report and submission.csv are not available until the official data are provided.')


Full empirical report and submission.csv are not available until the official data are provided.


## Sources

- Course handout: *Assignment 2: The Morning Rush*, supplied by the instructor, pp. 1–4.
- TA announcement, *Additional Notes for Assignment 2*, 19 October 2026 (text supplied by the user): https://e3p.nycu.edu.tw/mod/forum/discuss.php?d=135396
- Course Kaggle / submission announcement (text supplied by the user): Part III on Kaggle, 6 November 2026 23:59 deadline, 5 submissions/day, four official files including baseline.txt, and the updated late penalties. Invitation: https://www.kaggle.com/t/93a788c887e34269939b30da23305dfe
- Kaggle competition description (text supplied by the user): the data dictionary, hourly station-departure target, submission example `ID,Rentals`, and strict test-RMSE comparison with the official TA baseline. The downloaded sample submission remains authoritative for the actual file format.
- Official `A2_data_and_baseline_1_6` dataset supplied by the user: train.csv, test.csv, sample_submission.csv, and baseline.txt. The baseline file explicitly gives a TA leaderboard test-RMSE threshold of 24.2461; its May R² is retained as a quoted reference and distinguished from the recomputed value.
- User-supplied Kaggle leaderboard screenshot `image(20261022-071609).png`: team `415511001_嚴佳華`, score `12.37017`, rank 6 and 1 submission at the time of the screenshot; the page states that all test data are used. This is recorded evidence, not a live ranking query.
- scikit-learn Pipeline: https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html
- scikit-learn SGDRegressor: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDRegressor.html
- scikit-learn learning_curve: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.learning_curve.html

This notebook is an independent implementation based on the handout and the
TA clarification that the starter is optional. Synthetic program checks, if
performed during development, are excluded from formal assignment results.
